# Rediseño zero-shot multilingüe
### Proyecto Twitter/Tesis · WC2026 · notebook 2

El notebook 1 reprodujo el método actual. Este prueba el rediseño y **los compara sobre los
mismos textos**, para que la decisión quede respaldada con números y no con una intuición.

### Qué cambia

| | Método actual | Rediseño |
|---|---|---|
| Dimensiones | `hospitality`, `atmosphere`, `governance_trust` por listas de palabras en **en + es** | `atmosfera`, `imagen_cultural`, `perspectiva_politica` por zero-shot, **todos los idiomas** |
| Modelo de violencia | BART-MNLI (en/es) + XLM-R-XNLI (resto) | XLM-R-XNLI para **todo** |
| Valor 0 | ambiguo: ¿neutral o sin información? | se separa en `score` + `relevancia` |
| Escalón | ±0.2 por palabra encontrada | continuo |

### Por qué

Las listas de palabras solo cubren dos idiomas. En el corpus real —2 631 tuits, junio y julio
de 2026— eso significa que **432 tuits en japonés (16%) son invisibles por construcción**: no
hay una sola palabra japonesa en el léxico, así que sus tres dimensiones valen lo mismo pase
lo que pase. Un sesgo del instrumento confundido con la variable de agrupación.

Y no es solo cobertura. La sección 10 muestra que, en la dimensión política, el método actual
**invierte el signo** según el idioma: le da `+0.16` en inglés a un texto que dice que la
policía era corrupta —porque `police` y `authorities` están en la lista positiva— y `−0.16`
en español.

### Tres entradas

`FUENTE` en la sección 6 elige entre la **batería de prueba** (30 frases escritas a propósito,
por defecto), **textos propios** pegados a mano, y el **corpus** leído del CSV del proyecto.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
Sin GPU esto es impracticable: son 13 pasadas de un modelo *large* por tuit, y sobre el
corpus completo son 34 203.

## 1. Instalación

⚠️ Si la celda te pide reiniciar el entorno, hazlo (`Entorno de ejecución → Reiniciar sesión`)
y vuelve a correr desde aquí.

In [ ]:
import importlib.metadata as md_
import subprocess, sys

try:
    v = md_.version("transformers")
except md_.PackageNotFoundError:
    v = None
print("transformers instalado:", v)

necesita_reinicio = False
if v is None or int(v.split(".")[0]) >= 5:
    print("instalando transformers 4.x (rama del Dockerfile del proyecto)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5"],
        check=True,
    )
    necesita_reinicio = (v is not None)
else:
    print("version 4.x correcta, no se instala nada")

import torch
print("torch :", torch.__version__)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO -- sera MUY lento")

if necesita_reinicio:
    print()
    print("*" * 70)
    print("REINICIA EL ENTORNO AHORA: Entorno de ejecucion -> Reiniciar sesion")
    print("*" * 70)

## 2. Definición de las dimensiones

Aquí vive todo el diseño. Cada dimensión son **tres hipótesis**: positiva, negativa y una de
irrelevancia. El modelo reparte probabilidad entre las tres y de ahí salen dos números:

```
score      = P(positiva) − P(negativa)        va de −1 a 1
relevancia = 1 − P(irrelevante)               va de 0 a 1
```

La tercera hipótesis es la pieza clave. Sin ella, un tuit sobre fútbol preguntado por
gobierno reparte 50/50 y devuelve 0 — el mismo 0 ambiguo del método actual. Con ella,
`relevancia ≈ 0` te dice que el turista simplemente no habló del tema.

### Las dimensiones

| Dimensión | Qué mide |
|---|---|
| `atmosfera` | el ánimo que se vive: alegre y emocionante contra desagradable y estresante |
| `imagen_cultural` | comida, lugares y cultura: hermosos e impresionantes contra decepcionantes |
| `perspectiva_politica` | el gobierno y sus autoridades: competentes y legales contra corruptos |
| `violence_salience` · `safety_perceived` | violencia y seguridad, con las cuatro etiquetas y las dos fórmulas de `analyzer.py` |

### Dónde va el ancla «México»

La tesis mide la imagen de México, así que el instrumento tiene que estar anclado ahí. Pero
el ancla va en la **plantilla**, no dentro de cada hipótesis, y la diferencia no es cosmética.

`multi_label=False` reparte la probabilidad con `softmax` sobre los logits de las tres
etiquetas, y el softmax es invariante a un desplazamiento **constante**. Metida dentro de las
etiquetas, el ancla no es constante: encarece las dos afirmaciones (`pos`, `neg`) y **abarata
la negación** (`na`), porque es trivialmente cierto que un texto no menciona el ánimo *en
México* si no menciona México. El resultado es masa desplazada hacia `na`, relevancia hundida
y una alerta de «dimensión muda» causada por la redacción y no por el corpus.

En la plantilla el ancla es un prefijo idéntico para las tres etiquetas y se cancela.

**No es una suposición:** `ANCLA_EN_ETIQUETAS` genera la variante alterna a partir de las
mismas frases, y la sección 8 corre la batería con las dos y compara. Si el efecto no
aparece, la decisión se revisa con el número delante.

**Estas frases son el instrumento de medición.** Cambiarlas cambia los resultados, así que
en la tesis van citadas textualmente, igual que se cita un libro de códigos.

In [ ]:
# ===========================================================================
# EL INSTRUMENTO DE MEDICION. Todo el diseno vive aqui.
#
# Idioma de las hipotesis: XNLI es translinguistico, la hipotesis puede ir en
# un idioma y el tuit en otro. Lo importante es usar UNA SOLA para todos los
# textos, para que el instrumento sea identico entre idiomas.
# ===========================================================================
IDIOMA_HIPOTESIS = "es"

# ---------------------------------------------------------------------------
# DONDE VA EL ANCLA GEOGRAFICA. Es una decision de diseno, no un detalle de
# redaccion, y conviene entender por que.
#
# multi_label=False calcula softmax(logit_pos, logit_neg, logit_na) sobre los
# logits de entailment. El softmax es invariante a un desplazamiento CONSTANTE
# entre las tres etiquetas. El ancla, metida DENTRO de cada etiqueta, no lo es:
#
#   pos: "el ambiente EN MEXICO fue alegre"        afirmacion + predicado
#   neg: "el ambiente EN MEXICO fue desagradable"    -> mas dificil de entrañar
#   na : "NO menciona el ambiente ... EN MEXICO"   negacion + predicado
#                                                    -> MAS FACIL de entrañar
#
# Es trivialmente cierto que un texto no menciona el animo *en Mexico* si no
# menciona Mexico. El ancla empuja masa hacia `na`, hunde
# relevancia = 1 - P(na) bajo UMBRAL_MUDEZ y dispara la alerta de "dimension
# muda" por un artefacto de REDACCION, no por el corpus.
#
# En la PLANTILLA el ancla es un prefijo identico para las tres etiquetas y
# se cancela. Por eso ahi es donde va.
#
# ANCLA_EN_ETIQUETAS existe solo para MEDIR ese efecto en vez de suponerlo:
# la seccion de verificacion del ancla corre la bateria con las dos variantes
# y compara las distribuciones de relevancia. No lo dejes en True para una
# corrida de produccion.
# ---------------------------------------------------------------------------
ANCLA_EN_ETIQUETAS = False

_PLANTILLAS = {
    # ancla en la plantilla: constante entre etiquetas, se cancela
    False: {"en": "This text about Mexico says that {}.",
            "es": "Este texto sobre Mexico dice que {}."},
    # ancla dentro de las etiquetas: la plantilla queda neutra
    True:  {"en": "This text says that {}.",
            "es": "Este texto dice que {}."},
}
PLANTILLA = _PLANTILLAS[ANCLA_EN_ETIQUETAS][IDIOMA_HIPOTESIS]

# ---------------------------------------------------------------------------
# LAS DIMENSIONES. Cada una son tres hipotesis que compiten: positiva,
# negativa y de irrelevancia.
#
#   score      = P(pos) - P(neg)     va de -1 a 1
#   relevancia = 1 - P(na)           va de 0 a 1
#
# Las frases van SIN "en Mexico": el ancla la pone la plantilla. La variante
# anclada se deriva abajo, mecanicamente, para la comparacion.
#
# ESTAS FRASES SON EL INSTRUMENTO. Cambiarlas cambia los resultados, asi que
# en la tesis van citadas textualmente, igual que se cita un libro de codigos.
# ---------------------------------------------------------------------------
DIMENSIONES = {
    "atmosfera": {
        "en": {"pos": "the atmosphere was joyful and exciting for visitors",
               "neg": "the atmosphere was unpleasant and stressful for visitors",
               "na":  "it does not mention the atmosphere or the mood experienced"},
        "es": {"pos": "el ambiente fue alegre y emocionante para los visitantes",
               "neg": "el ambiente fue desagradable y estresante para los visitantes",
               "na":  "no menciona el ambiente ni el estado de animo que se vive"},
    },
    "imagen_cultural": {
        "en": {"pos": "the food, landmarks and culture are beautiful and impressive",
               "neg": "the food, landmarks and culture are disappointing and unappealing",
               "na":  "it does not mention the food, landmarks or culture"},
        "es": {"pos": "la comida, los lugares y la cultura son hermosos e impresionantes",
               "neg": "la comida, los lugares y la cultura son decepcionantes y poco atractivos",
               "na":  "no menciona la comida, los lugares ni la cultura"},
    },
    "perspectiva_politica": {
        "en": {"pos": "the government and its authorities acted in a competent, lawful and fair way",
               "neg": "the government and its authorities acted in a corrupt, criminal and incompetent way",
               "na":  "it does not mention the government, its politicians or its authorities"},
        "es": {"pos": "el gobierno y sus autoridades actuaron de forma competente, legal y justa",
               "neg": "el gobierno y sus autoridades actuaron de forma corrupta, criminal e incompetente",
               "na":  "no menciona al gobierno, a sus politicos ni a sus autoridades"},
    },
}

# ---------------------------------------------------------------------------
# Grupo de violencia/seguridad: se conservan las CUATRO etiquetas y las dos
# formulas del codigo actual (analyzer.py L67), de modo que violence_salience
# y safety_perceived se sigan calculando igual. Respecto del notebook previo
# lo unico que cambia es donde vive el ancla; el texto es el mismo.
# ---------------------------------------------------------------------------
VIOLENCIA = {
    "en": {"violence":   "there was violence or crime",
           "insecurity": "the place felt unsafe or insecure",
           "safety":     "the place felt safe and calm",
           "neutral":    "it does not talk about safety or violence"},
    "es": {"violence":   "hubo violencia o crimen",
           "insecurity": "el lugar se sentia inseguro",
           "safety":     "el lugar se sentia seguro y tranquilo",
           "neutral":    "no habla de seguridad ni violencia"},
}

# ---------------------------------------------------------------------------
# Variante con el ancla DENTRO de las etiquetas. Se DERIVA de las frases de
# arriba en vez de escribirse a mano, para que las dos variantes no puedan
# divergir en el contenido: la unica diferencia entre ellas es el ancla.
# ---------------------------------------------------------------------------
_ANCLA = {"en": " in Mexico", "es": " en Mexico"}


def _con_ancla(etiquetas, idi):
    return {k: v + _ANCLA[idi] for k, v in etiquetas.items()}


if ANCLA_EN_ETIQUETAS:
    DIMENSIONES = {d: {idi: _con_ancla(t[idi], idi) for idi in t}
                   for d, t in DIMENSIONES.items()}
    VIOLENCIA = {idi: _con_ancla(VIOLENCIA[idi], idi) for idi in VIOLENCIA}

# --- resumen ---------------------------------------------------------------
_ETIQ = {d: len(t[IDIOMA_HIPOTESIS]) for d, t in DIMENSIONES.items()}
_ETIQ["violencia"] = len(VIOLENCIA[IDIOMA_HIPOTESIS])

print("idioma de las hipotesis :", IDIOMA_HIPOTESIS)
print("ancla geografica        :", "DENTRO de las etiquetas (variante de prueba)"
      if ANCLA_EN_ETIQUETAS else "en la plantilla (se cancela en el softmax)")
print("plantilla               :", PLANTILLA)
print()
print("dimensiones             :", len(DIMENSIONES))
for _d, _t in DIMENSIONES.items():
    print("   {:<22} {} hipotesis".format(_d, len(_t[IDIOMA_HIPOTESIS])))
    for _k, _v in _t[IDIOMA_HIPOTESIS].items():
        print("      {:<4} {}".format(_k, PLANTILLA.format(_v)))
print("   {:<22} {} etiquetas".format("violencia/seguridad", _ETIQ["violencia"]))
for _k, _v in VIOLENCIA[IDIOMA_HIPOTESIS].items():
    print("      {:<11} {}".format(_k, PLANTILLA.format(_v)))
print()
print("pasadas de NLI por texto:", sum(_ETIQ.values()),
      " (" + " + ".join("{}={}".format(k, v) for k, v in _ETIQ.items()) + ")")

## 3. Carga de modelos

Dos modelos, ~3.3 GB en la máquina de Google. **No se usa BART**: el rediseño lo reemplaza
por XLM-R-XNLI, que es multilingüe de verdad. (BART-MNLI está entrenado solo en inglés, así
que el código actual ya manda los tuits en español a un modelo fuera de su distribución.)

In [ ]:
from transformers import pipeline
import torch

DEVICE = 0 if torch.cuda.is_available() else -1

print("1/2 modelo de sentimiento (~1.1 GB)...")
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True, max_length=128, device=DEVICE,
)

print("2/2 modelo NLI multilingue (~2.2 GB, tarda)...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    truncation=True, device=DEVICE,
)
print("listo")

## 4. Método actual (para comparar)

Copiado literal de `analyzer.py`. Solo necesita el modelo de sentimiento: los tres
indicadores léxicos son conteo de palabras puro.

In [ ]:
import math, re

def clamp(x, lo, hi):
    return min(max(x, lo), hi)

# --- analyzer.py L15-24 ----------------------------------------------------
# Copiadas literalmente, incluidas las de hospitalidad: el rediseno ya no mide
# esa dimension, pero la lista se conserva para que esta celda siga siendo una
# reproduccion fiel del codigo actual y se vea que no se toco nada.
HOSPITALITY_POS = {"friendly", "warm", "kind", "welcoming", "helpful", "amable", "hospitalario", "gentil"}
HOSPITALITY_NEG = {"rude", "hostile", "unfriendly", "aggressive", "grosero"}
ATMOSPHERE_POS = {"amazing", "incredible", "beautiful", "electric", "vibrant", "festive", "lively", "increíble", "hermoso", "ambiente"}
ATMOSPHERE_NEG = {"depressing", "unsafe", "scary", "tense", "chaotic", "deprimiente"}
GOVERNANCE_POS = {"police", "security", "safe", "protected", "policia", "seguridad", "officers", "authorities", "controlled"}
GOVERNANCE_NEG = {"corrupt", "impunity", "cartel", "crime", "criminal", "corrupción", "impunidad", "narco"}

# Correspondencia entre las dimensiones que mide el rediseno y las listas de
# palabras del metodo actual.
#
#   atmosfera            <- ATMOSPHERE_*    (analyzer.py las llama atmosphere)
#   perspectiva_politica <- GOVERNANCE_*    (analyzer.py: governance_trust)
#   imagen_cultural      <- NO EXISTE. Por eso ESCALAS la declara con un solo
#                           metodo y las figuras comparativas la excluyen.
#
# Las listas NO se tocan: renombrar la dimension no cambia que palabras busca
# el metodo actual, y el punto de la comparacion es medirlo tal como es.
SETS_LEXICO = {
    "atmosfera":            (ATMOSPHERE_POS, ATMOSPHERE_NEG),
    "perspectiva_politica": (GOVERNANCE_POS, GOVERNANCE_NEG),
}

def sentimiento(text):
    r = sentiment_pipe(text[:512])[0]
    lab = r["label"].lower()
    if lab == "positive":
        return r["score"], r["label"]
    if lab == "negative":
        return -r["score"], r["label"]
    return 0.0, r["label"]

def _mod(tl, pos_set, neg_set):
    pos = sorted(w for w in pos_set if w in tl)
    neg = sorted(w for w in neg_set if w in tl)
    return clamp((len(pos) - len(neg)) * 0.2, -1, 1), pos, neg

def metodo_actual(text):
    """analyzer.py L212-227 -- las dimensiones lexicas + sentimiento.

    Las FORMULAS son las del repositorio, intactas. Lo unico que cambia
    respecto del notebook anterior son las claves con que se devuelven, para
    que coincidan con las dimensiones que mide el rediseno.
    """
    tl = text.lower()
    s, lab = sentimiento(text)
    ma, ap, an = _mod(tl, ATMOSPHERE_POS,  ATMOSPHERE_NEG)
    mg, gp, gn = _mod(tl, GOVERNANCE_POS,  GOVERNANCE_NEG)
    return {
        "overall_sentiment":    s,
        "atmosfera":            clamp(s * 0.7 + ma, -1, 1),
        "perspectiva_politica": clamp(mg * 0.8, -1, 1),
        "_label": lab,
        "_palabras": {"atmosfera": ap + an, "perspectiva_politica": gp + gn},
    }

print("metodo actual listo --", len(SETS_LEXICO), "dimensiones lexicas:",
      ", ".join(SETS_LEXICO))

## 5. Rediseño zero-shot

Una llamada al modelo NLI por dimensión. `multi_label=False` hace que las tres hipótesis
compitan y sumen 1, que es lo que permite restarlas.

In [ ]:
IDI = IDIOMA_HIPOTESIS

# ===========================================================================
# EVALUACION POR LOTES
#
# Con 30 textos da igual. Con el corpus son 34 203 pasadas de un modelo large,
# y hacerlas de a una es la diferencia entre minutos y horas.
#
# BATCH_SIZE vive en la celda de configuracion, que corre DESPUES de esta. Se
# lee en el momento de la llamada, no al definir la funcion, y con un valor de
# respaldo por si alguien ejecuta esta celda suelta.
# ===========================================================================

def _batch():
    return globals().get("BATCH_SIZE", 8)


def _nli_lote(textos, etiquetas):
    """Una llamada por grupo de hipotesis para TODOS los textos del lote.

    Devuelve una lista de {etiqueta_legible: probabilidad}, una por texto, en
    el mismo orden de entrada. La forma por texto es identica a la que
    devolvia la version de a uno, asi que nada de lo que cuelga de aqui cambia.
    """
    frases = list(etiquetas.values())
    claves = list(etiquetas.keys())
    entrada = [t[:512] for t in textos]
    rs = nli_pipe(entrada, candidate_labels=frases,
                  hypothesis_template=PLANTILLA, multi_label=False,
                  batch_size=_batch())
    if isinstance(rs, dict):          # el pipeline devuelve un dict si le dan uno solo
        rs = [rs]
    salida = []
    for r in rs:
        por_frase = dict(zip(r["labels"], r["scores"]))
        salida.append({k: por_frase[etiquetas[k]] for k in claves})
    return salida


def _nli(text, etiquetas):
    """Compatibilidad: un solo texto."""
    return _nli_lote([text], etiquetas)[0]


def _sentimiento_lote(textos):
    rs = sentiment_pipe([t[:512] for t in textos], batch_size=_batch())
    if isinstance(rs, dict):
        rs = [rs]
    salida = []
    for r in rs:
        lab = r["label"].lower()
        if lab == "positive":
            salida.append((r["score"], r["label"]))
        elif lab == "negative":
            salida.append((-r["score"], r["label"]))
        else:
            salida.append((0.0, r["label"]))
    return salida


def metodo_nuevo_lote(textos, retweets=None, likes=None):
    """El rediseno sobre una lista de textos. Formulas identicas a las de la
    version de a uno; lo unico que cambia es que las llamadas van agrupadas."""
    n = len(textos)
    retweets = retweets if retweets is not None else [0] * n
    likes = likes if likes is not None else [0] * n
    if n == 0:
        return []

    sents = _sentimiento_lote(textos)
    salidas = []
    for i in range(n):
        s, lab = sents[i]
        salidas.append({"overall_sentiment": s, "_label": lab,
                        "engagement_weight": math.log1p(retweets[i] + likes[i]),
                        "_crudo": {}})

    # violencia / seguridad -- 4 etiquetas, formulas del repo
    for i, v in enumerate(_nli_lote(textos, VIOLENCIA[IDI])):
        o = salidas[i]
        o["_crudo"]["violencia"] = v
        o["violence_salience"] = clamp(v["violence"] + 0.5 * v["insecurity"], 0, 1)
        o["safety_perceived"] = clamp(v["safety"] - v["violence"], -1, 1)

    # las dimensiones de tres hipotesis
    for dim, etiquetas in DIMENSIONES.items():
        for i, p in enumerate(_nli_lote(textos, etiquetas[IDI])):
            o = salidas[i]
            o["_crudo"][dim] = p
            o[dim] = clamp(p["pos"] - p["neg"], -1, 1)
            o[dim + "_relevancia"] = clamp(1 - p["na"], 0, 1)

    return salidas


def metodo_nuevo(text, retweets=0, likes=0):
    """Compatibilidad: un solo texto."""
    return metodo_nuevo_lote([text], [retweets], [likes])[0]


_pasadas = sum(len(t[IDI]) for t in DIMENSIONES.values()) + len(VIOLENCIA[IDI])
print("rediseno listo --", _pasadas, "pasadas de NLI por tuit, en lotes de",
      "BATCH_SIZE (se lee al llamar)")

## 6. Configuración de las figuras

Los umbrales con los que se leen las figuras, todos en un solo lugar.

El importante es `UMBRAL_MUDEZ`. Marca dónde dejamos de leer un `score` como juicio y
empezamos a leerlo como «no habló del tema». **Es arbitrario**, y por eso la figura de
ausencia dibuja la distribución completa de `relevancia` con el corte superpuesto, en vez
de esconderlo: así se ve qué tan sensible es la lectura al valor elegido. En la tesis este
corte se justifica o se somete a análisis de sensibilidad; no se hereda de aquí.

También se declara aquí la **escala de cada dimensión** —valencia de −1 a 1, o prominencia
de 0 a 1— en vez de inferirla de los valores observados. Con 24 textos, concluir «esta
dimensión va de 0 a 1 porque no vi negativos» es exactamente el error que se quiere evitar.

In [ ]:
# ===========================================================================
# UMBRALES DE LECTURA. Ninguno es sagrado; todos se cambian aqui y solo aqui.
# ===========================================================================

# UMBRAL_MUDEZ es ARBITRARIO, y conviene decirlo en voz alta porque de el
# depende como se lee cada figura. Marca donde dejamos de leer un score como
# juicio y empezamos a leerlo como "no hablo del tema".
#
# En la tesis este corte hay que JUSTIFICARLO o someterlo a un analisis de
# sensibilidad; no se hereda de este notebook. Por eso la figura de ausencia
# dibuja la distribucion COMPLETA de relevancia con el corte superpuesto:
# para que se vea que tan sensible es la lectura al valor elegido.
UMBRAL_MUDEZ = 0.30

# Proporcion de observaciones mudas por encima de la cual una dimension se
# declara muda EN ESTE CORPUS y se dispara una alerta escrita. No dice que la
# dimension sea mala: dice que aqui casi nadie hablo de eso, y que por tanto
# su promedio esta dominado por textos que no opinaron.
UMBRAL_DIMENSION_MUDA = 0.70

# Por debajo de esto una figura se dibuja igual, pero con advertencia encima.
# Con la bateria de ejemplo (30 textos) varias figuras caen aqui, y es correcto
# que lo digan.
N_MINIMO_FIGURA = 20

# Minimo de observaciones para que un idioma se dibuje como comparable en las
# figuras que comparan idiomas. Por debajo se informa su n y se excluye; con
# el corpus real hay idiomas con una sola observacion.
N_MINIMO_IDIOMA = 3

# TODO lo que el notebook escribe va aqui, y nada de lo que lee. No es orden
# por gusto: con las salidas en el directorio de entrada, la autodeteccion del
# CSV encuentra dos archivos en la segunda corrida y el notebook se bloquea a
# si mismo.
SALIDA_DIR = "salida"

SALIDA_HTML = "figuras_dimensiones.html"

# CSV de resultados. El ancho trae UNA FILA POR TEXTO con las columnas
# originales del archivo mas una columna por dimension y metodo: es el que
# abres en Excel o le pasas a alguien. El largo trae una fila por texto,
# metodo y dimension: es el que se analiza.
SALIDA_CSV = "resultados_enriquecidos.csv"
SALIDA_CSV_LARGO = "resultados_largo.csv"
EXPORTAR_LARGO = True
# Probabilidades crudas de cada hipotesis como columnas. Son 13 columnas mas y
# hacen auditable cada valor: sin ellas hay que creerle a la resta.
EXPORTAR_PROBABILIDADES = True

# --- rutas compuestas, que es lo que usan las celdas de escritura ---------
import os as _os_cfg

_os_cfg.makedirs(SALIDA_DIR, exist_ok=True)
RUTA_HTML = _os_cfg.path.join(SALIDA_DIR, SALIDA_HTML)
RUTA_CSV_ANCHO = _os_cfg.path.join(SALIDA_DIR, SALIDA_CSV)
RUTA_CSV_LARGO = _os_cfg.path.join(SALIDA_DIR, SALIDA_CSV_LARGO)
CDN_ECHARTS = "https://cdnjs.cloudflare.com/ajax/libs/echarts/5.6.0/echarts.min.js"

# ===========================================================================
# DE DONDE SALEN LOS TEXTOS
#
#   "bateria"  la bateria de prueba del instrumento. No es una muestra de
#              nada: sirve para medir el instrumento, no el corpus.
#   "propios"  textos pegados a mano en la celda de la seccion 7.
#   "corpus"   el CSV del proyecto.
# ===========================================================================
FUENTE = "bateria"

# --- configuracion del corpus ----------------------------------------------
# RUTA_CSV = None autodetecta el unico .csv del directorio de trabajo. Si hay
# cero o mas de uno, la carga se detiene y los lista, en vez de elegir por ti.
RUTA_CSV = None

# Sobre que columna corre el MODELO. El metodo lexico corre sobre las dos
# (ver COMPARAR_TRADUCCION), porque es emparejamiento de strings y no cuesta
# pasadas de modelo.
COLUMNA_TEXTO = "text"

# El lexico sobre la traduccion responde la objecion "traduce todo y sigue con
# el lexico" sin consumir una sola pasada de modelo. OJO al interpretarlo:
# mide el lexico JUNTO CON la calidad de una traduccion automatica, y los dos
# efectos no se separan sin traducciones de referencia humanas.
COMPARAR_TRADUCCION = True

# Mapeo declarado, no adivinado. Se valida contra el encabezado real ANTES de
# cargar ningun modelo: si falta una columna, el error enumera las que si
# estan. Pon None en las opcionales que tu archivo no traiga.
COLUMNAS = {
    "texto":       "text",
    "traduccion":  "traducción",
    "tweet_id":    "id",
    "created_at":  "created_at",
    "lang":        "lang",
    "likes":       "like_count",
    "retweets":    "retweet_count",
    "estrato":     "base",          # con que consulta se recolecto
    "modalidad":   "corpus_type",   # volume / consolidated
    "es_retweet":  "is_retweet",
    "es_quote":    "is_quote",
}

# Tope de textos a evaluar. Pon None para el corpus completo. Empieza con 200,
# cronometra, extrapola, y solo entonces sueltalo: son 13 pasadas de un modelo
# large por texto.
N_MUESTRA = 200

# El corpus trae filas repetidas por haber sido recolectadas por mas de una
# consulta. Se deduplica por identificador: no hacerlo las cuenta dos veces en
# cada promedio.
DEDUPE = True

# Corre la bateria con las DOS variantes del ancla y compara las
# distribuciones de relevancia. Cuesta 13 pasadas extra por texto (390 sobre la
# bateria), asi que viene apagado. NO lo actives con el corpus.
# Apagado, la decision de poner el ancla en la plantilla queda solo por el
# argumento mecanico de la seccion 2, sin medicion que la respalde.
COMPARAR_ANCLA = False

# Textos por lote en las llamadas al modelo NLI. Con la bateria da igual; con
# el corpus es la diferencia entre minutos y horas. Bajalo si te quedas sin
# memoria de GPU.
BATCH_SIZE = 16

# Codigos con que Twitter marca "no se detecto idioma". No son idiomas y no se
# dibujan como si lo fueran: se agrupan en una categoria explicita.
CODIGOS_SIN_IDIOMA = {"und", "qme", "qam", "qst", "qct", "qht", "zxx", "art"}
ETIQUETA_SIN_IDIOMA = "sin_idioma"

# Que consultas buscaban expresamente que dimension. Sirve para avisar cuando
# un promedio mezcla estratos dirigidos con estratos que no lo son: promediar
# violencia sobre un corpus donde el 40% se busco POR violencia mide la
# consulta tanto como el fenomeno. Las claves se buscan en minusculas dentro
# del nombre del estrato.
CONSULTAS_DIRIGIDAS = {
    "violence_salience": ("victimizacion", "victimizacion_ja", "violencia", "crimen"),
    "safety_perceived":  ("victimizacion", "victimizacion_ja", "violencia", "seguridad"),
}

# ---------------------------------------------------------------------------
# La escala de cada dimension se DECLARA, no se infiere de los valores
# observados. Con 30 textos, inferir "esta dimension va de 0 a 1 porque no vi
# negativos" es exactamente el error que se quiere evitar.
#
#   valencia    -1..1   el cero es un juicio neutro
#   prominencia  0..1   el cero es ausencia
#
# relevancia : si la dimension trae su propio eje de ausencia (la 3a hipotesis)
# metodos    : que metodos la producen
# grupo      : con que llamada al modelo NLI se obtiene (None = no usa NLI)
#
# imagen_cultural declara UN SOLO metodo: no existe lista de palabras
# equivalente en analyzer.py. Las figuras que comparan metodos la excluyen a
# partir de esta declaracion y lo explican, en vez de descubrirlo por ausencia
# de datos y dibujar un hueco.
# ---------------------------------------------------------------------------
_LEXICOS = ("actual", "traducido") if COMPARAR_TRADUCCION else ("actual",)

ESCALAS = {
    "atmosfera":            {"escala": "valencia",    "relevancia": True,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": "atmosfera"},
    "imagen_cultural":      {"escala": "valencia",    "relevancia": True,
                             "metodos": ("nuevo",),   "grupo": "imagen_cultural"},
    "perspectiva_politica": {"escala": "valencia",    "relevancia": True,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": "perspectiva_politica"},
    "violence_salience":    {"escala": "prominencia", "relevancia": False,
                             "metodos": ("nuevo",),   "grupo": "violencia"},
    "safety_perceived":     {"escala": "valencia",    "relevancia": False,
                             "metodos": ("nuevo",),   "grupo": "violencia"},
    "overall_sentiment":    {"escala": "valencia",    "relevancia": False,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": None},
}

DIMS_CON_RELEVANCIA = [d for d, s in ESCALAS.items() if s["relevancia"]]

# Metodos que son emparejamiento de palabras, y sobre que columna corre cada
# uno. El rediseno lee siempre la columna que declare COLUMNA_TEXTO.
METODOS_LEXICOS = {"actual": "texto", "traducido": "traduccion"}

# ---------------------------------------------------------------------------
# Donde el metodo ACTUAL es un escalon puro.
#
#   perspectiva_politica = clamp(mg * 0.8)   con   mg = clamp((pos - neg) * 0.2)
#
# No incorpora el sentimiento, asi que solo puede tomar 11 valores discretos.
# Eso tiene una consecuencia util: su valor es EXACTO y reproducible sin GPU,
# que es lo que convierte la demostracion de la seccion 10 en un numero citable
# en vez de en el resultado de una corrida.
#
# atmosfera = clamp(s*0.7 + ma) SI incorpora el sentimiento, y el sentimiento
# es continuo: no es escalonada, es una nube desplazada a saltos. Marcar
# valores discretos ahi seria dibujar algo falso.
# ---------------------------------------------------------------------------
ESCALONADAS = {
    "perspectiva_politica": sorted({
        round(min(max(k * 0.2, -1.0), 1.0) * 0.8, 4) for k in range(-5, 6)
    })
}

# --- validacion de la configuracion ----------------------------------------
if FUENTE not in ("bateria", "propios", "corpus"):
    raise ValueError(
        "FUENTE = {!r} no es valido. Usa 'bateria', 'propios' o 'corpus'.".format(FUENTE))

_dims_sin_grupo = [d for d, s in ESCALAS.items()
                   if s["grupo"] and s["grupo"] not in DIMENSIONES and s["grupo"] != "violencia"]
if _dims_sin_grupo:
    raise ValueError(
        "ESCALAS declara grupos que no existen en DIMENSIONES: {}. "
        "Las dos celdas tienen que hablar del mismo conjunto.".format(_dims_sin_grupo))

_dims_sin_escala = [d for d in DIMENSIONES if d not in ESCALAS]
if _dims_sin_escala:
    raise ValueError(
        "DIMENSIONES define {} pero ESCALAS no las declara. "
        "Sin escala declarada ninguna figura sabe como dibujarlas.".format(_dims_sin_escala))

print("umbral de mudez         ", UMBRAL_MUDEZ, "  (arbitrario: justificar en la tesis)")
print("umbral dimension muda   ", UMBRAL_DIMENSION_MUDA)
print("n minimo por figura     ", N_MINIMO_FIGURA, "   por idioma:", N_MINIMO_IDIOMA)
print("dimensiones declaradas  ", len(ESCALAS), "  con relevancia:", len(DIMS_CON_RELEVANCIA))
for _d, _v in ESCALONADAS.items():
    print("escalonada: {:<22} {} valores posibles".format(_d, len(_v)))
_solo_nuevo = [d for d, s in ESCALAS.items() if s["metodos"] == ("nuevo",)]
print("solo rediseno           ", _solo_nuevo)
print()
print("fuente de textos        ", FUENTE)
if FUENTE == "corpus":
    print("   columna del modelo   ", COLUMNA_TEXTO)
    print("   lexico sobre traduccion:", COMPARAR_TRADUCCION)
    print("   tope de muestra      ", N_MUESTRA if N_MUESTRA else "sin tope")
    print("   deduplicar por id    ", DEDUPE)
print("tamano de lote          ", BATCH_SIZE)
print("las salidas van a        {}/  (nunca al directorio de entrada)".format(SALIDA_DIR))

## 7. Los textos a evaluar

`FUENTE`, en la sección 6, elige entre tres entradas.

### `"bateria"` — batería de prueba del instrumento (por defecto)

Cinco frases escritas a propósito, cada una en seis idiomas. 30 textos.

**No es una muestra de nada.** Es un instrumento de prueba: el contenido de las seis
versiones de una misma frase es el mismo, así que cualquier diferencia de medición entre
ellas es atribuible al instrumento y no al texto. Ningún número salido de aquí es
reportable como hallazgo sobre el corpus, y las figuras que salen de ella van rotuladas
como tales.

| Frase | Para qué está |
|---|---|
| `ambiente_pos` | única sonda positiva de `atmosfera`. **Es la frase nueva de este rediseño** |
| `imagen_pos` (comida) | única sonda de `imagen_cultural`, la dimensión sin método léxico |
| `politica_neg` | es el caso de la sección 10; reproduce `MISMA_FRASE` textualmente |
| `violencia` | única vía a `violence_salience` y `safety_perceived` |
| **`control_mudo`** (la gente fue amable) | no habla de ninguna de las tres dimensiones |

El control mudo es el que carga el peso. Sin él no hay forma de demostrar que el eje de
relevancia funciona: es el tuit que debe dar `relevancia ≈ 0` en las tres, y **no** un
juicio neutro.

**Ojo con la reasignación.** El texto de los tacos era el control mudo del diseño anterior.
Con `imagen_cultural` en el conjunto dejó de serlo —hablar de comida *es* hablar del tema—,
así que ascendió a sonda positiva de esa dimensión, y el control mudo pasó a ser la frase
sobre el trato de la gente, que no toca ánimo, ni comida o lugares, ni autoridades.

### `"propios"` — textos pegados a mano

Pega tus textos en la lista de la celda de abajo. La validación de relleno sigue en pie: si
dejas alguno sin sustituir, no corre.

### `"corpus"` — el CSV del proyecto

Lee el archivo del directorio de trabajo (autodetectado si hay uno solo). El mapeo de
columnas se declara en la sección 6 y **se valida contra el encabezado antes de cargar
ningún modelo**: si falta una columna, el error enumera las que sí están.

Empieza con `N_MUESTRA = 200`, cronometra, extrapola, y solo entonces suéltalo. Son 13
pasadas de un modelo *large* por texto.

In [ ]:
# ===========================================================================
# BATERIA DE PRUEBA DEL INSTRUMENTO
#
# NO ES UNA MUESTRA DE NADA. Son cinco frases escritas a proposito, cada una
# en los seis idiomas de la prueba. El contenido de las seis versiones de una
# misma frase es EL MISMO, de modo que cualquier diferencia de medicion entre
# ellas es atribuible al instrumento y no al texto. Ningun numero salido de
# aqui es reportable como hallazgo sobre el corpus; las figuras generadas
# sobre la bateria van rotuladas como tales.
#
# TRADUCCIONES: generadas por el asistente durante el diseno del notebook,
# SIN revision de hablante nativo. Hay que declararlo en la tesis. Si una
# version resulta mas entusiasta que otra, parte de la dispersion que se
# atribuye al instrumento es artefacto de traduccion.
#   Atenuante: eso pesa sobre la columna del REDISENO. La dispersion del
#   METODO ACTUAL no depende de la calidad de la traduccion sino de la
#   AUSENCIA de las palabras en el lexico, y eso se verifica a simple vista
#   en la columna "palabras que activo", que sale vacia.
#
# REASIGNACION. Al cambiar el conjunto de dimensiones, cuatro de las cinco
# frases se reutilizan tal cual y solo una es nueva:
#
#   politica_neg    era gobernanza_neg     -> perspectiva_politica + seccion 10
#   imagen_pos      era el control mudo    -> imagen_cultural  (habla de comida)
#   control_mudo    era hospitalidad_pos   -> control mudo del trio nuevo
#   violencia       igual                  -> violence_salience
#   ambiente_pos    NUEVA                  -> atmosfera
#
# El texto de comida dejaba de ser mudo de todas formas: con imagen_cultural
# en el conjunto, hablar de tacos es hablar del tema. Y "la gente fue amable"
# no habla de animo, ni de comida o lugares, ni de autoridades: sirve de
# control mudo del trio nuevo conservando sus seis versiones.
#
# Las seis versiones de politica_neg son las mismas de MISMA_FRASE (seccion 10)
# a proposito: la seccion 10 imprime su dispersion y la figura la recalcula
# desde aqui, asi que ambas tienen que coincidir. La celda de figuras lo
# comprueba y falla si alguien edita una y no la otra.
# ===========================================================================

BATERIA = {
    "ambiente_pos": {
        "objetivo": "atmosfera",
        "que_prueba": "juicio positivo explicito sobre el animo que se vive; unica sonda positiva de atmosfera",
        "nueva": True,
        "textos": {
            "en": "The atmosphere in the streets was festive and lively, I couldn't stop smiling all day.",
            "es": "El ambiente en las calles fue festivo y animado, no paré de sonreír en todo el día.",
            "pt": "O clima nas ruas era festivo e animado, não parei de sorrir o dia todo.",
            "fr": "L'ambiance dans les rues était festive et animée, je n'ai pas arrêté de sourire de la journée.",
            "de": "Die Stimmung auf den Straßen war festlich und ausgelassen, ich habe den ganzen Tag gelächelt.",
            "ar": "كان الجو في الشوارع احتفاليًا ومفعمًا بالحياة، ولم أتوقف عن الابتسام طوال اليوم.",
        },
    },
    "imagen_pos": {
        "objetivo": "imagen_cultural",
        "que_prueba": "habla de comida y de moverse por la ciudad; unica sonda de imagen_cultural, que no tiene metodo lexico equivalente",
        "nueva": False,
        "textos": {
            "en": "I had three tacos al pastor for breakfast and then took the metro downtown.",
            "es": "Desayuné tres tacos al pastor y luego tomé el metro al centro.",
            "pt": "Tomei três tacos al pastor no café da manhã e depois peguei o metrô para o centro.",
            "fr": "J'ai mangé trois tacos al pastor au petit-déjeuner puis j'ai pris le métro au centre-ville.",
            "de": "Ich habe drei Tacos al Pastor zum Frühstück gegessen und dann die U-Bahn ins Zentrum genommen.",
            "ar": "تناولت ثلاثة تاكو آل باستور على الإفطار ثم ركبت المترو إلى وسط المدينة.",
        },
    },
    "politica_neg": {
        "objetivo": "perspectiva_politica",
        "que_prueba": "juicio negativo explicito sobre autoridades; es la dimension donde el metodo actual es escalon puro y donde INVIERTE EL SIGNO entre idiomas (seccion 10)",
        "nueva": False,
        "textos": {
            "en": "The police were corrupt and did nothing, the authorities failed us completely.",
            "es": "La policía fue corrupta y no hizo nada, las autoridades nos fallaron por completo.",
            "pt": "A polícia era corrupta e não fez nada, as autoridades nos falharam completamente.",
            "fr": "La police était corrompue et n'a rien fait, les autorités nous ont complètement laissés tomber.",
            "de": "Die Polizei war korrupt und tat nichts, die Behörden haben uns völlig im Stich gelassen.",
            "ar": "كانت الشرطة فاسدة ولم تفعل شيئًا، وخذلتنا السلطات تمامًا.",
        },
    },
    "violencia": {
        "objetivo": "violence_salience",
        "que_prueba": "unica via a violence_salience y safety_perceived",
        "nueva": False,
        "textos": {
            "en": "There were gunshots near the stadium last night, people ran in panic.",
            "es": "Hubo disparos cerca del estadio anoche, la gente corrió presa del pánico.",
            "pt": "Houve tiros perto do estádio ontem à noite, as pessoas correram em pânico.",
            "fr": "Il y a eu des coups de feu près du stade hier soir, les gens ont couru pris de panique.",
            "de": "Gestern Abend fielen Schüsse in der Nähe des Stadions, die Menschen rannten in Panik.",
            "ar": "سُمع إطلاق نار قرب الملعب الليلة الماضية، وركض الناس في حالة ذعر.",
        },
    },
    "control_mudo": {
        "objetivo": None,
        "que_prueba": "NO habla del animo que se vive, ni de comida, lugares o cultura, ni de autoridades. Es la prueba del eje de relevancia: debe dar relevancia baja en las tres, no un juicio neutro",
        "nueva": False,
        "textos": {
            "en": "The people here were incredibly welcoming, I felt at home the whole trip.",
            "es": "La gente aquí fue increíblemente amable, me sentí como en casa todo el viaje.",
            "pt": "As pessoas aqui foram incrivelmente acolhedoras, me senti em casa a viagem toda.",
            "fr": "Les gens ici ont été incroyablement accueillants, je me suis senti chez moi tout le voyage.",
            "de": "Die Menschen hier waren unglaublich gastfreundlich, ich fühlte mich die ganze Reise wie zu Hause.",
            "ar": "كان الناس هنا مرحبين بشكل لا يصدق، شعرت وكأنني في بيتي طوال الرحلة.",
        },
    },
}

TRADUCCION_POR = "asistente (automatica), sin revision de hablante nativo"


# ===========================================================================
# TRES ENTRADAS, UNA SOLA FORMA DE SALIDA
#
# Pase lo que pase aqui, abajo quedan siempre:
#
#   TUITS  [(texto, retweets, likes), ...]     lo que consume la comparacion
#   META   [{texto_id, frase, lang, objetivo, tweet_id, created_at,
#            estrato, modalidad, traduccion}, ...]   en la MISMA posicion
#
# La procedencia viaja aparte, en META, para que TUITS conserve la forma que
# ya esperaba la celda de comparacion.
# ===========================================================================

def _meta_vacia(texto_id):
    return {"texto_id": texto_id, "frase": None, "lang": None, "objetivo": None,
            "tweet_id": None, "created_at": None, "estrato": None,
            "modalidad": None, "traduccion": None, "es_retweet": None,
            "es_quote": None}


def _cargar_bateria():
    tuits, meta = [], []
    for frase, spec in BATERIA.items():
        for lang, texto in spec["textos"].items():
            tuits.append((texto, 0, 0))
            m = _meta_vacia("{}:{}".format(frase, lang))
            m.update(frase=frase, lang=lang, objetivo=spec["objetivo"])
            meta.append(m)
    return tuits, meta


def _cargar_propios():
    # TUS TUITS AQUI. Sustituye los textos de relleno; la validacion de abajo
    # no te deja correr con ellos puestos.
    tuits = [
        # ("texto del tuit", retweets, likes)
        ("PEGA AQUI TU PRIMER TUIT", 0, 0),
        ("PEGA AQUI TU SEGUNDO TUIT", 0, 0),
    ]
    meta = [_meta_vacia("tuit_{:03d}".format(i)) for i in range(len(tuits))]
    relleno = [t for t, _, _ in tuits if t.startswith("PEGA AQUI")]
    if relleno:
        raise ValueError(
            "Todavia hay {} texto(s) de relleno en TUITS. Reemplazalos por tus "
            "tuits reales, o pon FUENTE = 'bateria' para correr con la bateria "
            "de ejemplo.".format(len(relleno)))
    return tuits, meta


def _localizar_csv():
    import glob, os
    if RUTA_CSV:
        if not os.path.exists(RUTA_CSV):
            raise FileNotFoundError(
                "RUTA_CSV = {!r} no existe. En Colab sube el archivo al panel "
                "de archivos, o deja RUTA_CSV = None para autodetectarlo."
                .format(RUTA_CSV))
        return RUTA_CSV
    # Los CSV que ESTE notebook escribe no son candidatos a entrada. Sin esto,
    # la segunda corrida encuentra dos archivos y se detiene pidiendo RUTA_CSV:
    # el notebook se bloquearia a si mismo en cuanto exporta una vez.
    # Las salidas viven en SALIDA_DIR, asi que normalmente aqui no hay nada
    # que excluir. La exclusion se queda como segundo cinturon por si alguien
    # pone SALIDA_DIR = "." y vuelve a mezclar entrada con salida.
    _propios = {os.path.basename(p).lower()
                for p in (globals().get("SALIDA_CSV"),
                          globals().get("SALIDA_CSV_LARGO")) if p}
    candidatos = [p for p in sorted(glob.glob("*.csv"))
                  if os.path.basename(p).lower() not in _propios]
    if len(candidatos) == 1:
        return candidatos[0]
    if not candidatos:
        raise FileNotFoundError(
            "No hay ningun .csv de entrada en el directorio de trabajo ({}). "
            "Sube el corpus al panel de archivos de Colab, o declara RUTA_CSV. "
            "(Los archivos que este notebook exporta no cuentan como entrada.)"
            .format(os.getcwd()))
    raise ValueError(
        "Hay {} archivos .csv y no se cual quieres: {}. Declara RUTA_CSV."
        .format(len(candidatos), candidatos))


def _a_bool(v):
    """'False', 'FALSE' y 'false' son el mismo valor. En el corpus conviven."""
    if v is None:
        return None
    s = str(v).strip().lower()
    if s in ("true", "1", "yes", "si", "sí"):
        return True
    if s in ("false", "0", "no", ""):
        return False
    return None


CORPUS_DF = None       # marco de origen, solo cuando FUENTE == "corpus"
CORPUS_RUTA = None


def _cargar_corpus():
    import pandas as pd

    ruta = _localizar_csv()
    # utf-8-sig: el archivo trae marca de orden de bytes y sin esto la primera
    # columna se llamaria '﻿base' y el mapeo fallaria por una razon
    # invisible al leer el error.
    df = pd.read_csv(ruta, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    print("archivo      :", ruta)
    print("filas leidas :", len(df), " columnas:", len(df.columns))

    # --- validacion del mapeo ANTES de tocar ningun modelo -----------------
    faltan = [k for k, c in COLUMNAS.items() if c and c not in df.columns]
    if faltan:
        raise KeyError(
            "El mapeo COLUMNAS pide columnas que el archivo no tiene: {}.\n"
            "Las {} columnas presentes son:\n  {}\n"
            "Corrige COLUMNAS en la celda de configuracion, o pon None en las "
            "que tu archivo no traiga."
            .format({k: COLUMNAS[k] for k in faltan}, len(df.columns),
                    "\n  ".join(sorted(df.columns))))

    col_modelo = COLUMNA_TEXTO
    if col_modelo not in df.columns:
        raise KeyError(
            "COLUMNA_TEXTO = {!r} no existe en el archivo. Columnas: {}"
            .format(col_modelo, sorted(df.columns)))

    n0 = len(df)
    # --- filas sin texto ---------------------------------------------------
    vacias = df[col_modelo].astype(str).str.strip() == ""
    if vacias.any():
        print("sin texto    :", int(vacias.sum()), "filas descartadas")
        df = df[~vacias]

    # --- deduplicacion -----------------------------------------------------
    col_id = COLUMNAS.get("tweet_id")
    if DEDUPE and col_id:
        antes = len(df)
        df = df.drop_duplicates(subset=[col_id], keep="first")
        if antes != len(df):
            print("duplicados   :", antes - len(df),
                  "filas con el mismo", col_id, "descartadas")

    # --- muestreo ----------------------------------------------------------
    if N_MUESTRA and len(df) > N_MUESTRA:
        print("muestra      :", N_MUESTRA, "de", len(df),
              "({} omitidos: sube N_MUESTRA o ponlo en None)".format(len(df) - N_MUESTRA))
        df = df.head(N_MUESTRA)

    # --- informes sobre lo que falta, en vez de fallar en silencio ---------
    col_fecha = COLUMNAS.get("created_at")
    if col_fecha:
        sin_fecha = (df[col_fecha].astype(str).str.strip() == "").sum()
        if sin_fecha:
            print("sin fecha    :", int(sin_fecha),
                  "filas cargadas con la fecha marcada como no disponible")
    col_trad = COLUMNAS.get("traduccion")
    if COMPARAR_TRADUCCION and col_trad:
        sin_trad = (df[col_trad].astype(str).str.strip() == "").sum()
        if sin_trad:
            print("sin traducir :", int(sin_trad),
                  "filas quedan fuera de la via traducida (no se sustituyen "
                  "por el original)")

    def _num(v):
        try:
            return int(float(str(v).strip() or 0))
        except (TypeError, ValueError):
            return 0

    def _opt(fila, clave):
        c = COLUMNAS.get(clave)
        if not c:
            return None
        v = str(fila.get(c, "") or "").strip()
        return v or None

    # Los booleanos del archivo mezclan capitalizaciones ('False' y 'FALSE'
    # conviven, seguramente por un lote fusionado de otra fuente). Sin
    # normalizar, un value_counts los cuenta como dos categorias distintas.
    for _clave in ("es_retweet", "es_quote"):
        _c = COLUMNAS.get(_clave)
        if _c:
            _vals = sorted(set(df[_c].astype(str).str.strip()))
            if len(_vals) > 2:
                print("booleanos    :", _c, "trae", _vals,
                      "-> se normalizan sin distinguir mayusculas")

    # Se conserva el marco de origen: la celda de exportacion devuelve el
    # archivo ENTERO con las columnas de resultados pegadas, y para eso hacen
    # falta las 52 columnas originales, no solo las que el modelo consume.
    global CORPUS_DF, CORPUS_RUTA
    CORPUS_DF, CORPUS_RUTA = df.copy(), ruta

    tuits, meta = [], []
    for pos, (_, fila) in enumerate(df.iterrows()):
        texto = str(fila[col_modelo])
        tuits.append((texto,
                      _num(fila.get(COLUMNAS.get("retweets"), 0)) if COLUMNAS.get("retweets") else 0,
                      _num(fila.get(COLUMNAS.get("likes"), 0)) if COLUMNAS.get("likes") else 0))
        tid = _opt(fila, "tweet_id")
        m = _meta_vacia(tid or "fila_{:05d}".format(pos))
        m.update(tweet_id=tid,
                 created_at=_opt(fila, "created_at"),
                 lang=_opt(fila, "lang"),
                 estrato=_opt(fila, "estrato"),
                 modalidad=_opt(fila, "modalidad"),
                 traduccion=_opt(fila, "traduccion"),
                 es_retweet=_a_bool(_opt(fila, "es_retweet")),
                 es_quote=_a_bool(_opt(fila, "es_quote")))
        meta.append(m)
    print("cargados     :", len(tuits), "textos de", n0, "filas del archivo")
    return tuits, meta


# --- construccion ----------------------------------------------------------
if FUENTE == "bateria":
    TUITS, META = _cargar_bateria()
elif FUENTE == "propios":
    TUITS, META = _cargar_propios()
elif FUENTE == "corpus":
    TUITS, META = _cargar_corpus()
else:
    raise ValueError("FUENTE = {!r} no es valido".format(FUENTE))

# Rotulo de procedencia de TRES estados. Antes era binario (bateria si/no) y
# ya no alcanza: el rotulo del corpus no dice "no reportable" sino algo
# distinto y mas facil de pasar por alto -- DESCRIPTIVO PERO NO VALIDADO,
# porque no hay codificacion humana contra la cual contrastar.
PROCEDENCIA = FUENTE
ES_BATERIA = (FUENTE == "bateria")

# --- costo de la corrida, derivado y no escrito a mano ---------------------
# El numero de pasadas sale del numero de etiquetas de cada grupo de
# hipotesis: zero-shot con multi_label=False hace una pasada por etiqueta.
_ETIQUETAS_POR_GRUPO = {"violencia": len(VIOLENCIA[IDIOMA_HIPOTESIS])}
for _d, _t in DIMENSIONES.items():
    _ETIQUETAS_POR_GRUPO[_d] = len(_t[IDIOMA_HIPOTESIS])
PASADAS_POR_TEXTO = sum(_ETIQUETAS_POR_GRUPO.values())

_ROTULO = {"bateria": "BATERIA DE PRUEBA -- no es una muestra de nada",
           "propios": "textos propios",
           "corpus":  "CORPUS -- descriptivo, NO validado contra codificacion humana"}

print()
print("{} textos listos  ({})".format(len(TUITS), _ROTULO[PROCEDENCIA]))
if ES_BATERIA:
    print("   {} frases x {} idiomas".format(
        len(BATERIA), len(next(iter(BATERIA.values()))["textos"])))
    print("   traducciones:", TRADUCCION_POR)
    _nuevas = [f for f, s in BATERIA.items() if s.get("nueva")]
    if _nuevas:
        print("   redactadas para este rediseno:", ", ".join(_nuevas))
if PROCEDENCIA == "corpus":
    _idiomas = {}
    for _m in META:
        _idiomas[_m["lang"] or "?"] = _idiomas.get(_m["lang"] or "?", 0) + 1
    print("   idiomas: " + ", ".join("{}={}".format(k, v) for k, v in
                                     sorted(_idiomas.items(), key=lambda p: -p[1])[:8]))
    _estratos = {}
    for _m in META:
        _estratos[_m["estrato"] or "?"] = _estratos.get(_m["estrato"] or "?", 0) + 1
    print("   estratos: {} distintos".format(len(_estratos)))
print()
print("costo de la corrida")
print("   pasadas de NLI por texto: {}   {}".format(
    PASADAS_POR_TEXTO,
    " + ".join("{}={}".format(k, v) for k, v in _ETIQUETAS_POR_GRUPO.items())))
print("   total:                    {} pasadas de un modelo large".format(
    len(TUITS) * PASADAS_POR_TEXTO))

try:
    import torch as _torch
    if not _torch.cuda.is_available():
        print()
        print("*" * 74)
        print("SIN GPU. Son {} pasadas de un modelo large; en CPU esto puede ser".format(
            len(TUITS) * PASADAS_POR_TEXTO))
        print("impracticable. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4)")
        print("*" * 74)
except Exception:
    pass

In [ ]:
# Dimensiones que tienen metodo lexico, derivadas de ESCALAS en vez de escritas
# a mano: si manana una dimension gana o pierde metodo, esto se entera solo.
LEX = [d for d, s in ESCALAS.items()
       if "actual" in s["metodos"] and d != "overall_sentiment"]

# Metodos lexicos realmente disponibles en esta corrida. "traducido" solo
# existe si hay columna de traduccion poblada.
_HAY_TRADUCCION = any(m.get("traduccion") for m in META)
METODOS_ACTIVOS = ["actual"]
if COMPARAR_TRADUCCION and _HAY_TRADUCCION:
    METODOS_ACTIVOS.append("traducido")


def _texto_de(metodo, texto, meta):
    """Sobre que cadena corre cada metodo lexico.

    La via traducida NO cae de vuelta al original cuando falta la traduccion:
    devuelve None y la fila se omite. Sustituirla en silencio mezclaria dos
    mediciones distintas bajo la misma etiqueta.
    """
    if metodo == "traducido":
        return meta.get("traduccion")
    return texto


# ===========================================================================
# EVALUACION
#
# El rediseno corre POR LOTES: una llamada al modelo por grupo de hipotesis
# para todos los textos, en vez de 13 llamadas por texto. Con la bateria da
# igual; con el corpus es la diferencia entre minutos y horas.
#
# El metodo lexico corre despues, texto por texto, porque es emparejamiento de
# strings. Lo unico que cuesta de la via traducida es UNA pasada del modelo de
# SENTIMIENTO (base, ~1.1 GB) por texto: ninguna pasada de NLI, que es el
# modelo large y el que domina el costo.
# ===========================================================================
import time

_t0 = time.perf_counter()

_textos = [t for t, _, _ in TUITS]
_rts = [rt for _, rt, _ in TUITS]
_lks = [lk for _, _, lk in TUITS]

print("evaluando {} textos con el rediseno (lotes de {})...".format(
    len(_textos), BATCH_SIZE))
_nuevos = metodo_nuevo_lote(_textos, _rts, _lks)

print("evaluando el metodo lexico sobre: {}".format(", ".join(METODOS_ACTIVOS)))
resultados = []
_sin_traduccion = 0
for _i, (_t, _rt, _lk) in enumerate(TUITS):
    _fila = {"nuevo": _nuevos[_i]}
    for _m in METODOS_ACTIVOS:
        _src = _texto_de(_m, _t, META[_i])
        if _src is None:
            _sin_traduccion += 1
            _fila[_m] = None
        else:
            _fila[_m] = metodo_actual(_src)
    resultados.append(_fila)

TIEMPO_CORRIDA = time.perf_counter() - _t0

if _sin_traduccion:
    print("   {} textos sin traduccion quedan fuera de la via traducida "
          "(no se sustituyen por el original)".format(_sin_traduccion))


# ===========================================================================
# IMPRESION. Se lee de `resultados`, ya calculado: imprimir no vuelve a
# llamar a ningun modelo.
# ===========================================================================
def comparar(i):
    r = resultados[i]
    texto, rt, lk = TUITS[i]
    a, n = r["actual"], r["nuevo"]
    print("=" * 88)
    print("TUIT: " + texto[:200].replace("\n", " "))
    print("-" * 88)
    print("  sentimiento = {:+.4f}  (label crudo {!r})".format(
        a["overall_sentiment"], a["_label"]))
    print()
    _cols = ["{:>10}".format(m.upper()) for m in METODOS_ACTIVOS]
    print("  {:<22}{} {:>10} {:>12} {:>20}".format(
        "dimension", "".join(_cols), "NUEVO", "relevancia", "palabras que activo"))
    print("  " + "-" * 86)
    for d in LEX:
        vals = ""
        for m in METODOS_ACTIVOS:
            vals += ("{:>+10.4f}".format(r[m][d]) if r[m] else "{:>10}".format("--"))
        pal = a["_palabras"][d]
        print("  {:<22}{} {:>+10.4f} {:>12.2f} {:>20}".format(
            d, vals, n[d], n[d + "_relevancia"], str(pal) if pal else "(ninguna)"))
    for d in ESCALAS:
        if d in LEX or d == "overall_sentiment" or d not in n:
            continue
        _rel = n.get(d + "_relevancia")
        print("  {:<22}{} {:>+10.4f} {:>12}".format(
            d, "{:>10}".format("--") * len(METODOS_ACTIVOS), n[d],
            "{:.2f}".format(_rel) if _rel is not None else "--"))
    print()
    print("  probabilidades crudas del modelo NLI:")
    for grupo, probs in n["_crudo"].items():
        detalle = "  ".join("{}={:.3f}".format(k, v) for k, v in probs.items())
        print("     {:<22} {}".format(grupo, detalle))
    print()
    return a, n


# Con el corpus no se imprimen 2 600 bloques: solo una muestra.
_A_IMPRIMIR = range(len(TUITS)) if PROCEDENCIA == "bateria" else range(min(5, len(TUITS)))
for _i in _A_IMPRIMIR:
    comparar(_i)
if len(_A_IMPRIMIR) < len(TUITS):
    print("... {} bloques mas no impresos (son {} textos). Los marcos y las "
          "figuras los tienen todos.".format(len(TUITS) - len(_A_IMPRIMIR), len(TUITS)))

print("=" * 88)
print("{} textos  ·  {} pasadas de NLI  ·  {:.1f} s  ({:.3f} s/texto)".format(
    len(TUITS), len(TUITS) * PASADAS_POR_TEXTO, TIEMPO_CORRIDA,
    TIEMPO_CORRIDA / max(len(TUITS), 1)))
if PROCEDENCIA != "corpus" or (N_MUESTRA and N_MUESTRA < 2000):
    print("Cronometra esto contra el tamano de tu corpus antes de comprometerte.")
    print("   a este ritmo, 2 631 textos tardarian {:.1f} min".format(
        TIEMPO_CORRIDA / max(len(TUITS), 1) * 2631 / 60))
print("=" * 88)

## 8. Verificación del ancla geográfica

La sección 2 afirma que meter «en México» **dentro** de cada hipótesis desplaza probabilidad
hacia `na` y hunde la relevancia. El argumento es mecánico —el softmax cancela un prefijo
constante, no un predicado que encarece las afirmaciones y abarata la negación— pero un
argumento no es una medición.

Esta sección lo mide. Corre la batería **dos veces**, con el ancla en la plantilla y con el
ancla en las etiquetas, y compara las distribuciones de relevancia. La variante alterna se
**deriva** de las mismas frases, así que no pueden diferir en nada más que en el ancla.

Cuesta 13 pasadas extra por texto, así que viene apagada: pon `COMPARAR_ANCLA = True` en la
sección 6. Sobre la batería son 390 pasadas más; no la actives con el corpus.

**Si el efecto no aparece, la decisión hay que revisarla.** Esta celda existe para poder
decirlo con el número delante, no para confirmar lo que ya se escribió.

In [ ]:
# ===========================================================================
# A/B DEL ANCLA GEOGRAFICA
#
# Mide lo que la seccion 2 afirma, en vez de darlo por bueno. La variante
# alterna se DERIVA de las mismas frases (misma funcion _con_ancla que usa la
# celda de dimensiones), asi que la unica diferencia entre las dos corridas es
# donde vive el ancla.
#
# Resultado -> COMPARACION_ANCLA, que la celda de figuras embebe.
# ===========================================================================
COMPARACION_ANCLA = None

if not COMPARAR_ANCLA:
    print("A/B del ancla desactivado (COMPARAR_ANCLA = False).")
    print("La decision de poner el ancla en la plantilla queda por el argumento")
    print("mecanico de la seccion 2, SIN medicion que la respalde. Actívalo para")
    print("respaldarla o para refutarla.")
else:
    _otra = not ANCLA_EN_ETIQUETAS

    def _variante(en_etiquetas):
        """Devuelve (plantilla, dimensiones, violencia) para una variante."""
        pl = _PLANTILLAS[en_etiquetas][IDIOMA_HIPOTESIS]
        # se parte SIEMPRE de las frases sin anclar, vengan como vengan
        base_d = {d: {idi: dict(t[idi]) for idi in t} for d, t in DIMENSIONES.items()}
        base_v = {idi: dict(VIOLENCIA[idi]) for idi in VIOLENCIA}
        if ANCLA_EN_ETIQUETAS:
            # las globales YA vienen ancladas: se les quita para tener la base
            _n = len(_ANCLA[IDIOMA_HIPOTESIS])
            base_d = {d: {idi: {k: v[:-_n] if v.endswith(_ANCLA[idi]) else v
                                for k, v in t[idi].items()} for idi in t}
                      for d, t in base_d.items()}
            base_v = {idi: {k: v[:-_n] if v.endswith(_ANCLA[idi]) else v
                            for k, v in base_v[idi].items()} for idi in base_v}
        if en_etiquetas:
            base_d = {d: {idi: _con_ancla(t[idi], idi) for idi in t}
                      for d, t in base_d.items()}
            base_v = {idi: _con_ancla(base_v[idi], idi) for idi in base_v}
        return pl, base_d, base_v

    def _correr_variante(en_etiquetas, textos):
        global PLANTILLA, DIMENSIONES, VIOLENCIA
        _guardado = (PLANTILLA, DIMENSIONES, VIOLENCIA)
        PLANTILLA, DIMENSIONES, VIOLENCIA = _variante(en_etiquetas)
        try:
            return metodo_nuevo_lote(textos)
        finally:
            PLANTILLA, DIMENSIONES, VIOLENCIA = _guardado

    _txt = [t for t, _, _ in TUITS]
    print("corriendo la bateria con las DOS variantes ({} textos x 2)...".format(len(_txt)))
    _res = {"plantilla": _correr_variante(False, _txt),
            "etiquetas": _correr_variante(True, _txt)}

    _dims = [d for d in DIMENSIONES if d in ESCALAS and ESCALAS[d]["relevancia"]]
    _filas = []
    for dim in _dims:
        fila = {"dimension": dim}
        for var, salidas in _res.items():
            rels = [o[dim + "_relevancia"] for o in salidas]
            media = sum(rels) / len(rels)
            fila[var] = {
                "relevancia_media": round(media, 4),
                "relevancia_min": round(min(rels), 4),
                "relevancia_max": round(max(rels), 4),
                "prop_muda": round(sum(1 for r in rels if r < UMBRAL_MUDEZ) / len(rels), 4),
                "dimension_muda": bool(
                    sum(1 for r in rels if r < UMBRAL_MUDEZ) / len(rels) > UMBRAL_DIMENSION_MUDA),
            }
        fila["delta_relevancia"] = round(
            fila["etiquetas"]["relevancia_media"] - fila["plantilla"]["relevancia_media"], 4)
        _filas.append(fila)

    _mudas_pl = sum(1 for f in _filas if f["plantilla"]["dimension_muda"])
    _mudas_et = sum(1 for f in _filas if f["etiquetas"]["dimension_muda"])
    _delta = sum(f["delta_relevancia"] for f in _filas) / len(_filas)
    # El efecto predicho: el ancla en las etiquetas infla na, luego BAJA la
    # relevancia. delta negativo = la prediccion se cumple.
    _confirmado = _delta < 0

    COMPARACION_ANCLA = {
        "filas": _filas,
        "delta_medio": round(_delta, 4),
        "dimensiones_mudas": {"plantilla": _mudas_pl, "etiquetas": _mudas_et},
        "prediccion_confirmada": bool(_confirmado),
        "umbral_mudez": UMBRAL_MUDEZ,
    }

    print()
    print("=" * 80)
    print("RELEVANCIA SEGUN DONDE VIVA EL ANCLA")
    print("=" * 80)
    print("  {:<22} {:>12} {:>12} {:>10}   {}".format(
        "dimension", "plantilla", "etiquetas", "delta", "% mudo (plant / etiq)"))
    print("  " + "-" * 78)
    for f in _filas:
        print("  {:<22} {:>12.4f} {:>12.4f} {:>+10.4f}   {:>5.0%} / {:<5.0%}".format(
            f["dimension"], f["plantilla"]["relevancia_media"],
            f["etiquetas"]["relevancia_media"], f["delta_relevancia"],
            f["plantilla"]["prop_muda"], f["etiquetas"]["prop_muda"]))
    print()
    print("  delta medio de relevancia: {:+.4f}".format(_delta))
    print("  dimensiones declaradas mudas:  plantilla {} · etiquetas {}".format(
        _mudas_pl, _mudas_et))
    print()
    if _confirmado:
        print("  PREDICCION CONFIRMADA. El ancla dentro de las etiquetas baja la")
        print("  relevancia, que es lo que la seccion 2 argumenta: infla `na` porque")
        print("  anadir el predicado a una negacion la hace mas facil de entrañar.")
        print("  La decision de ponerla en la plantilla queda respaldada por el numero.")
    else:
        print("  " + "!" * 76)
        print("  PREDICCION NO CONFIRMADA. El ancla dentro de las etiquetas NO baja la")
        print("  relevancia en esta corrida. El argumento mecanico de la seccion 2 no")
        print("  se sostiene tal como esta escrito, y la decision de diseño hay que")
        print("  REVISARLA antes de reportar nada. No lo tapes: documentalo.")
        print("  " + "!" * 76)
    print("=" * 80)

## 9. Marcos de datos

Un solo punto de extracción. Se recorre `resultados` una vez y salen tres marcos:

```
LARGO    (texto_id, metodo, dimension)  →  valencia, relevancia, escala
PROBS    (texto_id, grupo, etiqueta)    →  probabilidad
LEXICO   (texto_id, dimension, palabra) →  polaridad, es_subcadena
```

Tres claves distintas quieren tres marcos. Forzar uno solo obligaría a nulos estructurales:
la relevancia no existe para el método actual, las probabilidades crudas cuelgan del **grupo
de llamada NLI** y no de la dimensión reportada, y las palabras del léxico no tienen
equivalente en el rediseño.

Toda figura y toda tabla leen de aquí. Esto recupera lo que la tabla resumen descartaba
—`violence_salience`, `safety_perceived`, el sentimiento y las probabilidades crudas— y de
paso elimina la dependencia oculta que tenía de la sección 10.

**La guardia importante:** `valencia` y `relevancia` no se combinan nunca en un solo número.
El producto es tentador porque reduce todo a un eje, y reconstruye exactamente el defecto
que el rediseño elimina:

```
habló y juzgó neutro    0.02 × 0.91 = 0.018
no habló del tema       0.60 × 0.03 = 0.018
```

El mismo número para dos cosas distintas: el `0.0000` ambiguo con otro nombre. Hay una
aserción que falla si alguien agrega cualquier columna numérica nueva, que es la vía por la
que ese producto entraría sin que nadie lo note.

In [ ]:
import pandas as pd
import re

# ===========================================================================
# UN SOLO PUNTO DE EXTRACCION, TRES FORMAS
#
# Se recorre `resultados` una vez y salen tres marcos con claves limpias:
#
#   LARGO   (texto_id, metodo, dimension)  -> valencia, relevancia, escala
#   PROBS   (texto_id, grupo, etiqueta)    -> probabilidad
#   LEXICO  (texto_id, metodo, dimension, palabra) -> polaridad, es_subcadena
#
# Tres claves distintas quieren tres marcos. Forzar uno solo obligaria a
# nulos estructurales: la relevancia no existe para los metodos lexicos, las
# probabilidades crudas cuelgan del GRUPO de llamada NLI y no de la dimension
# reportada, y las palabras del lexico no tienen equivalente en el rediseno.
#
# La columna `metodo` admite MAS DE DOS valores: 'actual' (lexico sobre el
# texto original), 'traducido' (el mismo lexico sobre la traduccion) y
# 'nuevo'. Es un valor mas de una columna que ya existia, no una columna
# nueva: por eso las figuras lo recogen sin cambios.
#
# Toda figura y toda tabla leen de aqui. Nada vuelve a `resultados`.
# ===========================================================================

_LETRA = r"[^\W\d_]"


def es_subcadena(palabra, texto_lower):
    """True si la palabra SOLO aparece dentro de otra palabra mas larga.

    El metodo actual empareja con `w in tl`, sin frontera de palabra. Eso hace
    que 'safe' se active dentro de 'unsafe', 'corrupt' dentro de 'corrupta' y
    'kind' dentro de 'kinda'. Aqui no se corrige nada: se marca, para poder
    medir cuanto pesa.
    """
    patron = r"(?<!{L}){p}(?!{L})".format(L=_LETRA, p=re.escape(palabra))
    return re.search(patron, texto_lower) is None


def grupo_idioma(lang):
    """Los codigos con que Twitter marca 'no se detecto idioma' no son idiomas.

    Dibujarlos como si lo fueran inventa categorias; meterlos en un cajon de
    sastre sin nombre las esconde. Se agrupan bajo una etiqueta explicita.
    """
    if not lang:
        return None
    return ETIQUETA_SIN_IDIOMA if lang in CODIGOS_SIN_IDIOMA else lang


def construir_frames(resultados, tuits, meta):
    filas_valor, filas_prob, filas_lex = [], [], []

    for i, ((texto, rt, lk), r) in enumerate(zip(tuits, resultados)):
        m = meta[i]
        n = r["nuevo"]
        base = {
            "texto_id":   m["texto_id"],
            "frase":      m["frase"],
            "lang":       m["lang"],
            "lang_grupo": grupo_idioma(m["lang"]),
            "objetivo":   m["objetivo"],
            "texto":      texto,
            # Columnas de procedencia: vacias mientras la entrada sea una lista
            # de textos, pobladas cuando lo sea el corpus.
            "tweet_id":   m["tweet_id"],
            "created_at": m["created_at"],
            # Estratos de recoleccion. Sin estos, un promedio de violencia
            # sobre un corpus donde el 40% se busco POR violencia mide sobre
            # todo la consulta que lo recolecto.
            "estrato":    m["estrato"],
            "modalidad":  m["modalidad"],
            "es_retweet": m.get("es_retweet"),
            "es_quote":   m.get("es_quote"),
        }

        # --- valores por dimension y metodo --------------------------------
        for dim, spec in ESCALAS.items():
            for metodo in spec["metodos"]:
                src = r.get(metodo)
                if src is None or dim not in src:
                    continue
                # La relevancia NO existe para los metodos lexicos. Va como
                # NaN, nunca como 0: un 0 significaria "no hablo del tema",
                # que es justo la informacion que el lexico no tiene.
                relevancia = float("nan")
                if metodo == "nuevo" and spec["relevancia"]:
                    relevancia = float(n[dim + "_relevancia"])
                filas_valor.append(dict(
                    base,
                    metodo=metodo,
                    # Sobre que cadena se calculo esta fila. Sin esto, las dos
                    # vias lexicas serian indistinguibles en el marco.
                    origen_texto=METODOS_LEXICOS.get(metodo, "texto"),
                    dimension=dim,
                    escala=spec["escala"],
                    grupo=spec["grupo"],
                    valencia=float(src[dim]),
                    relevancia=relevancia,
                    engagement_weight=float(n["engagement_weight"]),
                    label_sentimiento=n["_label"],
                ))

        # --- probabilidades crudas, por grupo de llamada NLI ---------------
        # Un grupo puede producir mas de una dimension: 'violencia' tiene
        # cuatro etiquetas y de el salen violence_salience y safety_perceived.
        for grupo, probs in n["_crudo"].items():
            for etiqueta, p in probs.items():
                filas_prob.append({
                    "texto_id": base["texto_id"], "frase": base["frase"],
                    "lang": base["lang"], "lang_grupo": base["lang_grupo"],
                    "estrato": base["estrato"], "grupo": grupo,
                    "etiqueta": etiqueta, "probabilidad": float(p),
                })

        # --- palabras del lexico activadas ---------------------------------
        for metodo in METODOS_LEXICOS:
            src = r.get(metodo)
            if src is None:
                continue
            tl = (texto if metodo == "actual" else m["traduccion"]).lower()
            for dim, palabras in src["_palabras"].items():
                pos_set, neg_set = SETS_LEXICO[dim]
                for w in palabras:
                    filas_lex.append({
                        "texto_id": base["texto_id"], "frase": base["frase"],
                        "lang": base["lang"], "lang_grupo": base["lang_grupo"],
                        "metodo": metodo, "dimension": dim, "palabra": w,
                        "polaridad": "pos" if w in pos_set else "neg",
                        "es_subcadena": es_subcadena(w, tl),
                    })

    largo = pd.DataFrame(filas_valor)
    probs = pd.DataFrame(filas_prob)
    lexico = pd.DataFrame(filas_lex, columns=["texto_id", "frase", "lang",
                                              "lang_grupo", "metodo",
                                              "dimension", "palabra",
                                              "polaridad", "es_subcadena"])
    return largo, probs, lexico


LARGO, PROBS, LEXICO = construir_frames(resultados, TUITS, META)

# ---------------------------------------------------------------------------
# GUARDIA: valencia y relevancia no se combinan nunca en un escalar.
#
# El producto valencia * relevancia es tentador porque reduce todo a un eje,
# y reconstruye exactamente el defecto que el rediseno elimina:
#
#   hablo y juzgo neutro   0.02 * 0.91 = 0.018
#   no hablo del tema      0.60 * 0.03 = 0.018
#
# El mismo numero para dos cosas distintas: el 0.0000 ambiguo con otro nombre.
# Esta asercion falla si alguien agrega cualquier columna numerica nueva, que
# es la via por la que ese producto entraria sin que nadie lo note.
# ---------------------------------------------------------------------------
COLUMNAS_NUMERICAS_PERMITIDAS = {"valencia", "relevancia", "engagement_weight"}
_num = set(LARGO.select_dtypes("number").columns)
assert _num == COLUMNAS_NUMERICAS_PERMITIDAS, (
    "LARGO tiene columnas numericas inesperadas: {}. Si alguna combina valencia "
    "y relevancia en un solo numero, quitala: ese producto reintroduce el cero "
    "ambiguo.".format(sorted(_num - COLUMNAS_NUMERICAS_PERMITIDAS))
)

# ---------------------------------------------------------------------------
# IDIOMAS DEL CORPUS. Se derivan de los datos, no de una lista declarada.
#
# El notebook asumia seis grupos fijos. El corpus real tiene 23 valores de
# lang y CERO en aleman, neerlandes y arabe; el 16% es japones, un idioma que
# el diseno nunca contemplo. Declarar los grupos inventaria los vacios y
# esconderia el japones.
# ---------------------------------------------------------------------------
IDIOMAS_OBSERVADOS = (LARGO[["texto_id", "lang_grupo"]].drop_duplicates()
                      ["lang_grupo"].value_counts().to_dict()
                      if len(LARGO) and LARGO["lang_grupo"].notna().any() else {})
IDIOMAS_COMPARABLES = [l for l, n in IDIOMAS_OBSERVADOS.items() if n >= N_MINIMO_IDIOMA]
IDIOMAS_ESCASOS = {l: n for l, n in IDIOMAS_OBSERVADOS.items() if n < N_MINIMO_IDIOMA}

# Idiomas que el lexico no puede ver: ninguna de sus palabras esta en ese
# idioma. Sobre la bateria eso era una hipotesis; sobre el corpus es un numero.
_con_lexico = set(LEXICO["lang_grupo"].dropna().unique()) if len(LEXICO) else set()
IDIOMAS_SIN_COBERTURA = {l: n for l, n in IDIOMAS_OBSERVADOS.items()
                         if l not in _con_lexico}

print("=" * 74)
print("MARCOS CONSTRUIDOS")
print("=" * 74)
print("  LARGO   {:>6} filas   {} textos x {} dimensiones x metodo".format(
    len(LARGO), LARGO["texto_id"].nunique(), LARGO["dimension"].nunique()))
print("  PROBS   {:>6} filas   {} grupos de hipotesis".format(
    len(PROBS), PROBS["grupo"].nunique()))
print("  LEXICO  {:>6} filas   palabras del lexico activadas".format(len(LEXICO)))
print()
print("  metodos en el marco:", sorted(LARGO["metodo"].unique()))
for _m in sorted(LARGO["metodo"].unique()):
    _sub = LARGO[LARGO.metodo == _m]
    print("     {:<10} sobre {:<10} {} filas, {} dimensiones".format(
        _m, _sub["origen_texto"].iloc[0], len(_sub), _sub["dimension"].nunique()))
print()
_solo_uno = [d for d, s in ESCALAS.items() if len(s["metodos"]) == 1]
print("  dimensiones con un solo metodo (salen de las figuras comparativas):")
for _d in _solo_uno:
    print("     -", _d)
print()
_sin_rel = LARGO[(LARGO.metodo != "nuevo") & LARGO.relevancia.isna()]
print("  relevancia no disponible en {} filas de metodos lexicos (NaN, no 0)".format(
    len(_sin_rel)))
if len(LEXICO):
    _sub = int(LEXICO["es_subcadena"].sum())
    print("  emparejamientos lexicos: {} totales, {} por subcadena".format(
        len(LEXICO), _sub))
else:
    print("  emparejamientos lexicos: ninguno en este corpus")

if IDIOMAS_OBSERVADOS:
    print()
    print("  idiomas observados ({}):".format(len(IDIOMAS_OBSERVADOS)))
    for _l, _n in list(IDIOMAS_OBSERVADOS.items())[:12]:
        _marca = "" if _n >= N_MINIMO_IDIOMA else "  (bajo N_MINIMO_IDIOMA, se excluye)"
        print("     {:<12} {:>5} textos{}".format(_l, _n, _marca))
    if IDIOMAS_SIN_COBERTURA:
        _tot = sum(IDIOMAS_SIN_COBERTURA.values())
        print()
        print("  COBERTURA LEXICA NULA en {}: {} textos ({:.0%} del total).".format(
            sorted(IDIOMAS_SIN_COBERTURA), _tot, _tot / max(sum(IDIOMAS_OBSERVADOS.values()), 1)))
        print("  Para esos textos las dimensiones lexicas valen lo mismo POR")
        print("  CONSTRUCCION: no hay una sola palabra de esos idiomas en las listas.")

_estratos = LARGO["estrato"].dropna().nunique() if len(LARGO) else 0
if _estratos:
    print()
    print("  estratos de recoleccion: {} distintos".format(_estratos))
    for _e, _n in (LARGO[["texto_id", "estrato"]].drop_duplicates()
                   ["estrato"].value_counts().head(8).items()):
        print("     {:<42} {:>5} textos".format(str(_e)[:42], _n))
print("=" * 74)

## 10. La demostración del sesgo entre idiomas

Esta sección es el argumento central, y conviene que vaya en la tesis.

Abajo está **la misma frase** —contenido idéntico, sentido idéntico— escrita en los seis
idiomas de la batería: *la policía era corrupta y las autoridades fallaron*. Un instrumento
de medición válido debería darles aproximadamente el mismo valor, y **negativo**.

No los da. El método actual sale **positivo en inglés y en francés**, negativo en español y
portugués, y exactamente cero en alemán y árabe. La causa en inglés es fácil de ver en la
columna de palabras: `police` y `authorities` están en la lista **positiva** de `analyzer.py`,
así que un texto que denuncia a la policía suma dos positivas contra una negativa.

Eso no es imprecisión entre idiomas. Es la conclusión contraria según el idioma, y por eso
la celda lo señala aparte de la dispersión.

**Un detalle que hace fuerte a esta demostración:** `perspectiva_politica = clamp(mg × 0.8)`
no incorpora el sentimiento, así que su valor no depende de ningún modelo. Es aritmética
sobre listas de palabras. El resultado es exacto, reproducible sin GPU y citable como se
cita una cuenta, no como se cita un experimento.

Compara la columna ACTUAL contra la NUEVO, y mira también `relevancia`: es la información
que hoy no existe.

In [ ]:
# ===========================================================================
# LA MISMA FRASE EN SEIS IDIOMAS
#
# Son los mismos strings que BATERIA["politica_neg"], a proposito: esta celda
# imprime la dispersion y la celda de figuras la recalcula desde la bateria.
# La figura comprueba que coinciden y falla si alguien edita una y no la otra.
# ===========================================================================
MISMA_FRASE = [
    ("en", "The police were corrupt and did nothing, the authorities failed us completely."),
    ("es", "La policía fue corrupta y no hizo nada, las autoridades nos fallaron por completo."),
    ("pt", "A polícia era corrupta e não fez nada, as autoridades nos falharam completamente."),
    ("fr", "La police était corrompue et n'a rien fait, les autorités nous ont complètement laissés tomber."),
    ("de", "Die Polizei war korrupt und tat nichts, die Behörden haben uns völlig im Stich gelassen."),
    ("ar", "كانت الشرطة فاسدة ولم تفعل شيئًا، وخذلتنا السلطات تمامًا."),
]

DIMENSION_DEMO = "perspectiva_politica"

print("MISMA FRASE, SEIS IDIOMAS -- dimension '{}'".format(DIMENSION_DEMO))
print()
print("Contenido identico en las seis: la policia era corrupta y las autoridades")
print("fallaron. Un instrumento valido deberia darles el mismo valor, y NEGATIVO.")
print()
print("  {:<5} {:>10} {:>10} {:>12}   {}".format(
    "idioma", "ACTUAL", "NUEVO", "relevancia", "palabras del lexico"))
print("  " + "-" * 76)

filas_demo = []
_textos_demo = [f for _, f in MISMA_FRASE]
_nuevos_demo = metodo_nuevo_lote(_textos_demo)
for _i, (lang, frase) in enumerate(MISMA_FRASE):
    a = metodo_actual(frase)
    n = _nuevos_demo[_i]
    pal = a["_palabras"][DIMENSION_DEMO]
    print("  {:<5} {:>+10.4f} {:>+10.4f} {:>12.2f}   {}".format(
        lang, a[DIMENSION_DEMO], n[DIMENSION_DEMO],
        n[DIMENSION_DEMO + "_relevancia"], pal if pal else "(ninguna)"))
    filas_demo.append({"lang": lang, "actual": a[DIMENSION_DEMO],
                       "nuevo": n[DIMENSION_DEMO],
                       "relevancia": n[DIMENSION_DEMO + "_relevancia"],
                       "palabras": pal})


def _disp(vals):
    m = sum(vals) / len(vals)
    return max(vals) - min(vals), (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5


_va = [f["actual"] for f in filas_demo]
_vn = [f["nuevo"] for f in filas_demo]
_ra, _sa = _disp(_va)
_rn, _sn = _disp(_vn)

print()
print("  DISPERSION ENTRE IDIOMAS -- el numero que se reporta.")
print("  El contenido no varia, asi que toda variacion es del instrumento.")
print("  {:<10} rango {:>7.4f}   desv.est. {:>7.4f}".format("ACTUAL", _ra, _sa))
print("  {:<10} rango {:>7.4f}   desv.est. {:>7.4f}".format("NUEVO", _rn, _sn))

# ---------------------------------------------------------------------------
# INVERSION DE SIGNO. Es un resultado mas fuerte que la dispersion y merece
# senalarse aparte: no es que el instrumento sea impreciso entre idiomas, es
# que llega a la conclusion CONTRARIA segun el idioma.
#
# Ojo a la causa en ingles: 'police' y 'authorities' estan en la lista
# POSITIVA de analyzer.py, asi que un texto que dice que la policia era
# corrupta suma dos positivas y una negativa, y sale positivo.
#
# Este valor es EXACTO: perspectiva_politica = clamp(mg * 0.8) no incorpora el
# sentimiento, asi que no depende de ninguna corrida ni de ninguna GPU. Se
# puede citar en la tesis como se cita una cuenta, no como se cita un experimento.
# ---------------------------------------------------------------------------
def _signo(v, eps=1e-9):
    return 0 if abs(v) < eps else (1 if v > 0 else -1)


for _etiq, _vals in (("ACTUAL", _va), ("NUEVO", _vn)):
    _pos = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) > 0]
    _neg = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) < 0]
    _cero = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) == 0]
    if _pos and _neg:
        print()
        print("  " + "!" * 72)
        print("  INVERSION DE SIGNO en {}: el mismo contenido sale POSITIVO en {}".format(
            _etiq, ", ".join(_pos)))
        print("  y NEGATIVO en {}{}.".format(
            ", ".join(_neg),
            ", y exactamente cero en " + ", ".join(_cero) if _cero else ""))
        print("  No es imprecision entre idiomas: es la conclusion contraria.")
        print("  " + "!" * 72)
    elif _cero and (_pos or _neg):
        print()
        print("  {}: {} idioma(s) dan exactamente cero ({}), que aqui significa".format(
            _etiq, len(_cero), ", ".join(_cero)))
        print("  'el lexico no encontro ninguna palabra', no 'juicio neutro'.")

_sin_palabras = [f["lang"] for f in filas_demo if not f["palabras"]]
if _sin_palabras:
    print()
    print("  Sin ninguna palabra del lexico activada: {}. Para esos idiomas el".format(
        ", ".join(_sin_palabras)))
    print("  metodo actual no puede dar otra cosa que cero, pase lo que pase el texto.")

## 11. Tabla resumen

In [ ]:
import pandas as pd

# La tabla sale de LARGO, no de `resultados`. Antes se reconstruia a mano y en
# el camino perdia violence_salience, safety_perceived, el sentimiento y las
# probabilidades crudas. Ademas dependia de `filas_demo`, que nace en la
# seccion de demostracion entre idiomas: ejecutar esta celda sin haber corrido
# aquella reventaba con NameError. Ya no.

_corto = LARGO["texto"].str.slice(0, 38) + LARGO["texto"].str.len().gt(38).map({True: "...", False: ""})
_L = LARGO.assign(tuit=_corto)

# --- comparacion metodo actual contra rediseno -----------------------------
# Comparable = la produce el lexico Y el rediseno. Se pregunta por los metodos
# concretos y no por "cuantos hay": con la via traducida activa una dimension
# puede tener tres, y contar dejaria la tabla vacia sin avisar.
_comparables = [d for d, s in ESCALAS.items()
                if "actual" in s["metodos"] and "nuevo" in s["metodos"]]
_metodos_tabla = [m for m in ("actual", "traducido", "nuevo")
                  if m in set(LARGO["metodo"])]

# Los atributos por texto se unen DESPUES del pivote, no se meten en su indice:
# pivot_table descarta los grupos cuya clave es NaN, y con textos propios `lang`
# viene vacio. Meterlo en el indice vaciaria la tabla entera.
_attrs = _L[["texto_id", "tuit", "lang"]].drop_duplicates(subset=["texto_id"])

_piv = (_L[_L.dimension.isin(_comparables)]
        .pivot_table(index=["texto_id", "dimension"],
                     columns="metodo", values="valencia", aggfunc="first")
        .reset_index())

_rel = (_L[(_L.metodo == "nuevo")][["texto_id", "dimension", "relevancia"]]
        .drop_duplicates(subset=["texto_id", "dimension"]))

tabla = (_piv.merge(_attrs, on="texto_id", how="left")
             .merge(_rel, on=["texto_id", "dimension"], how="left"))
tabla["diferencia"] = tabla["nuevo"] - tabla["actual"]
_cols = ["tuit", "lang", "dimension"] + _metodos_tabla + ["relevancia", "diferencia"]
tabla = tabla[[c for c in _cols if c in tabla.columns]]
tabla = tabla.round({m: 3 for m in _metodos_tabla})
tabla = tabla.round({"relevancia": 2, "diferencia": 3})

print("COMPARACION ENTRE METODOS  (dimensiones que el lexico y el rediseno producen)")
if "traducido" in _metodos_tabla:
    print("  'traducido' es el MISMO lexico sobre la columna de traduccion. Mide el")
    print("  lexico junto con la calidad de una traduccion automatica: si gana, no")
    print("  se sabe cuanto merito es de cada uno.")
display(tabla)

# --- dimensiones que solo produce el rediseno ------------------------------
_solo_nuevo = [d for d, s in ESCALAS.items() if s["metodos"] == ("nuevo",)]
if _solo_nuevo:
    print()
    print("DIMENSIONES QUE SOLO EXISTEN EN EL REDISENO")
    print("(el metodo actual no las produce: no hay columna con que comparar.")
    print(" imagen_cultural esta aqui porque analyzer.py no tiene lista de")
    print(" palabras equivalente, no porque falten datos.)")
    _extra = (_L[_L.dimension.isin(_solo_nuevo)]
              .pivot_table(index="texto_id", columns="dimension",
                           values="valencia", aggfunc="first")
              .reset_index()
              .merge(_attrs, on="texto_id", how="left")
              .drop(columns=["texto_id"])
              .round(3))
    display(_extra)

# --- la seccion 10 sigue imprimiendo lo suyo; aqui solo se repite si existe --
if "filas_demo" in globals():
    print()
    print("Dispersion de la misma frase en 6 idiomas (seccion 10):")
    display(pd.DataFrame(filas_demo))

## 12. Figuras

Siete figuras, cada una respondiendo **una** pregunta, con su umbral y su alerta escrita —en
vez de dejar la conclusión al ojo.

| Figura | Pregunta |
|---|---|
| 1 · dispersión entre idiomas | ¿mide lo mismo cuando el contenido es el mismo? |
| 2 · valencia × relevancia | ¿cuántos de los ceros de hoy son «no habló»? |
| 3 · valores posibles | ¿qué valores puede tomar cada método? |
| 4 · ausencia por dimensión | ¿esta dimensión mide algo en este corpus? |
| 5 · descomposición `pos`/`neg`/`na` | ¿a dónde se va la probabilidad del modelo? |
| 6 · cobertura léxica | ¿en cuántos idiomas llega a activarse el léxico? |
| 7 · desglose por estrato | ¿de qué consulta viene cada número? |

**La ausencia nunca se apila con la escala.** `score` y `relevancia` son ejes ortogonales y
ninguna figura los colapsa. La banda de mudez se **marca**, no se filtra: dibujar esas
observaciones *es* la evidencia.

**Tres métodos, no dos.** Cuando el corpus trae traducción, el léxico corre sobre las dos
columnas y las figuras muestran `léxico (original)`, `léxico (traducido)` y `rediseño`. Se
distinguen por tono **y** por trama, para que sigan siendo legibles impresas en gris. Ojo al
leer la vía traducida: es el mismo léxico sobre otra entrada, así que mide el léxico **junto
con** la calidad de una traducción automática, y los dos efectos no se separan.

**`imagen_cultural` no aparece en las figuras 1, 3 ni 6.** No es que falten datos: no existe
lista de palabras equivalente en `analyzer.py`, así que no hay columna contra la cual
compararla. Las figuras lo dicen en vez de dibujar un hueco.

**Los idiomas salen de los datos.** El diseño anterior declaraba seis grupos fijos; el corpus
real tiene 23 valores de `lang`, cero en alemán, neerlandés y árabe, y un 16% en japonés que
nadie había contemplado. Los idiomas con menos de `N_MINIMO_IDIOMA` observaciones se nombran
y se excluyen, en vez de dibujarse como si fueran comparables.

**El rótulo de procedencia tiene tres estados.** Batería, textos propios y corpus. El del
corpus es el que importa: no dice «no reportable» sino **descriptivo pero no validado**, que
es más fácil de pasar por alto y más caro de olvidar.

**Salida doble.** Las figuras se muestran aquí abajo y además se escriben en un archivo HTML
autocontenido, que es el artefacto que sobrevive al cierre del notebook y va a la tesis. Cada
figura se exporta como SVG desde su barra de herramientas.

In [ ]:
import json
import datetime as _dt
import math
import pandas as pd
from IPython.display import HTML, display

# ===========================================================================
# PREPARACION DE DATOS
#
# Todo se calcula aqui, una vez, en Python. La pagina recibe numeros ya
# agregados y no necesita el kernel para redibujarse: por eso el archivo
# exportado sigue funcionando fuera del notebook.
#
# Las combinaciones sin observaciones se OMITEN, no se emiten como cero.
# Un cero significa "medi y dio cero"; la ausencia es otra cosa.
# ===========================================================================

PALETA = {
    # slots categoricos 1 y 2 del sistema de referencia; par adyacente
    # validado (CVD dE 24.7, vision normal dE 33.6, ambos >= umbral).
    "actual": "#2a78d6",
    "nuevo":  "#eb6834",
    # Tercer metodo: el MISMO lexico sobre la traduccion. Es hermano de
    # "actual", no un metodo independiente, asi que va como tinte claro del
    # mismo azul en vez de como un tercer matiz suelto. En escala de grises se
    # separa por luminosidad, y ademas lleva trama diagonal en la figura.
    "traducido": "#8fb8e8",
    # neutro deliberado para la AUSENCIA: falla el piso de croma del
    # validador a proposito, porque aqui "leerse gris" es el significado.
    "na":     "#898781",
    "neg":    "#e34948",
    "medio":  "#eda100",
    "tinta":  "#0b0b0b",
    "tinta2": "#52514e",
    "mudo":   "#898781",
    "grid":   "#e1e0d9",
    "eje":    "#c3c2b7",
    "sup":    "#fcfcfb",
    "alerta": "#d03b3b",
    "ok":     "#0ca30c",
}

_ALERTAS = []


def _alerta(texto):
    _ALERTAS.append(texto)


# --- comprobacion de coherencia con la seccion 10 ---------------------------
# La bateria reproduce textualmente las seis versiones de MISMA_FRASE. Si
# alguien edita una y no la otra, la figura de dispersion y lo que imprime la
# seccion 10 dejarian de coincidir en silencio. Aqui no.
if ES_BATERIA and "MISMA_FRASE" in globals():
    _b = BATERIA["politica_neg"]["textos"]
    _d = dict(MISMA_FRASE)
    _difieren = [k for k in _d if _b.get(k) != _d[k]]
    assert not _difieren, (
        "Los textos de politica_neg de la BATERIA y de MISMA_FRASE difieren en "
        "{}. Son los mismos a proposito: la seccion 10 imprime su dispersion y "
        "la figura la recalcula. Sincronizalos.".format(_difieren)
    )


def _stats(valores):
    v = [x for x in valores if x is not None and not (isinstance(x, float) and math.isnan(x))]
    if not v:
        return None
    media = sum(v) / len(v)
    var = sum((x - media) ** 2 for x in v) / len(v)
    return {"n": len(v), "rango": max(v) - min(v), "sigma": var ** 0.5, "media": media}


def preparar_datos():
    d = {
        "es_bateria": bool(ES_BATERIA),
        # Rotulo de TRES estados. El del corpus no dice "no reportable" sino
        # algo distinto y mas facil de pasar por alto: descriptivo pero NO
        # VALIDADO, porque no hay codificacion humana contra la cual contrastar.
        "procedencia": PROCEDENCIA,
        "rotulo": {
            "bateria": "BATERIA DE PRUEBA DEL INSTRUMENTO. No es una muestra de "
                       "nada: sus valores no son reportables como hallazgo sobre "
                       "el corpus.",
            "propios": "Textos aportados a mano. Ni bateria ni corpus: lo que "
                       "digan las figuras vale para estos textos y nada mas.",
            "corpus":  "CORPUS REAL. Las figuras son DESCRIPTIVAS y NO estan "
                       "validadas: el archivo no trae codificacion humana, asi "
                       "que el acuerdo contra codificadores sigue sin medirse.",
        }[PROCEDENCIA],
        "n_textos": int(LARGO["texto_id"].nunique()),
        "n_minimo": N_MINIMO_FIGURA,
        "umbral_mudez": UMBRAL_MUDEZ,
        "umbral_dim_muda": UMBRAL_DIMENSION_MUDA,
        "paleta": PALETA,
        # A/B del ancla: la evidencia de la decision de diseno de la seccion 2.
        # None cuando no se corrio, y la pagina lo dice en vez de callarlo.
        "ancla": globals().get("COMPARACION_ANCLA"),
        "ancla_en_etiquetas": bool(ANCLA_EN_ETIQUETAS),
    }

    # ---- ficha tecnica del instrumento ------------------------------------
    d["ficha"] = {
        "idioma_hipotesis": IDIOMA_HIPOTESIS,
        "plantilla": PLANTILLA,
        "modelo_sentimiento": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
        "modelo_nli": "joeddav/xlm-roberta-large-xnli",
        "fecha": _dt.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "pasadas_por_texto": PASADAS_POR_TEXTO,
        "tiempo_corrida_s": round(TIEMPO_CORRIDA, 1) if "TIEMPO_CORRIDA" in globals() else None,
        "traduccion": TRADUCCION_POR if ES_BATERIA else None,
        "hipotesis": [
            {"grupo": g, "etiqueta": k, "frase": v}
            for g, t in list(DIMENSIONES.items())
            for k, v in t[IDIOMA_HIPOTESIS].items()
        ] + [
            {"grupo": "violencia", "etiqueta": k, "frase": v}
            for k, v in VIOLENCIA[IDIOMA_HIPOTESIS].items()
        ],
    }

    # ---- textos de la bateria, para que las traducciones sean auditables --
    d["bateria"] = []
    if ES_BATERIA:
        for frase, spec in BATERIA.items():
            for lang, texto in spec["textos"].items():
                d["bateria"].append({"frase": frase, "lang": lang, "texto": texto,
                                     "objetivo": spec["objetivo"],
                                     "que_prueba": spec["que_prueba"]})

    # Los idiomas se DERIVAN de los datos. El notebook asumia seis grupos
    # fijos; el corpus real tiene 23 valores de lang, cero en aleman,
    # neerlandes y arabe, y un 16% en japones que el diseno nunca contemplo.
    # Declararlos inventaria los vacios y esconderia el japones.
    idiomas = [l for l in IDIOMAS_COMPARABLES]
    d["idiomas"] = idiomas
    d["idiomas_n"] = IDIOMAS_OBSERVADOS
    d["idiomas_escasos"] = IDIOMAS_ESCASOS
    d["idiomas_sin_cobertura"] = IDIOMAS_SIN_COBERTURA
    d["n_minimo_idioma"] = N_MINIMO_IDIOMA
    if IDIOMAS_ESCASOS:
        _alerta("IDIOMAS CON MUY POCAS OBSERVACIONES, excluidos de las figuras "
                "que comparan idiomas: {}. Se informan para que la exclusion sea "
                "visible, no para dibujarlos como si fueran comparables."
                .format(", ".join("{} (n={})".format(k, v)
                                  for k, v in sorted(IDIOMAS_ESCASOS.items()))))

    # ---- F1 · dispersion entre idiomas ------------------------------------
    d["dispersion"] = {}
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & LARGO.lang.notna()]
        if sub.empty or not idiomas:
            continue
        # Los metodos se leen de ESCALAS. Con la via traducida activa una
        # dimension puede tener tres, y fijar dos aqui la dejaria fuera.
        metodos = [m for m in ESCALAS[dim]["metodos"] if m in set(sub["metodo"])]
        filas = {}
        for metodo in metodos:
            s = sub[sub.metodo == metodo]
            # una frase por idioma: la que apunta a esta dimension si existe
            objetivo = s[s.objetivo == dim]
            s = objetivo if not objetivo.empty else s
            por_lang = s.groupby("lang_grupo")["valencia"].mean()
            filas[metodo] = [round(float(por_lang.get(l)), 4)
                             for l in idiomas if l in por_lang.index]
        langs = [l for l in idiomas
                 if l in sub.groupby("lang_grupo")["valencia"].mean().index]
        if not langs or len(metodos) < 2:
            continue
        d["dispersion"][dim] = {
            "langs": langs,
            "metodos": metodos,
            "series": {m: filas[m] for m in metodos},
            # se conservan las dos claves antiguas para no romper nada que las lea
            "actual": filas.get("actual", []),
            "nuevo": filas.get("nuevo", []),
            "stats": {m: _stats(filas[m]) for m in metodos},
        }
    # dimensiones que no pueden aparecer aqui, con su razon
    d["dispersion_excluidas"] = [
        {"dimension": dim,
         "razon": "solo la produce el rediseno: analyzer.py no tiene lista de "
                  "palabras equivalente, asi que no hay columna contra la cual "
                  "compararla. No es que falten datos."}
        for dim, sp in ESCALAS.items()
        if sp["metodos"] == ("nuevo",) and sp["relevancia"]
    ]

    # ---- F2 · plano valencia x relevancia ---------------------------------
    d["plano"] = {}
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & (LARGO.metodo == "nuevo")]
        puntos = {}
        for _, r in sub.iterrows():
            if pd.isna(r["relevancia"]):
                continue
            puntos.setdefault(r["frase"] or "sin frase", []).append({
                "x": round(float(r["relevancia"]), 4),
                "y": round(float(r["valencia"]), 4),
                "lang": r["lang"], "texto": r["texto"][:120],
            })
        if not puntos:
            continue
        mudos = int(sum(1 for g in puntos.values() for p in g
                        if p["x"] < UMBRAL_MUDEZ))
        d["plano"][dim] = {"grupos": puntos, "n": sum(len(g) for g in puntos.values()),
                           "mudos": mudos}
    # dimensiones excluidas de esta figura, con su razon
    d["plano_excluidas"] = [
        {"dimension": dim, "razon": "no produce relevancia: sale del grupo '{}', "
                                    "que no tiene hipotesis de irrelevancia propia "
                                    "por dimension".format(ESCALAS[dim]["grupo"])}
        for dim in ESCALAS if not ESCALAS[dim]["relevancia"] and dim != "overall_sentiment"
    ]

    # ---- F3 · valores posibles de cada metodo -----------------------------
    d["valores"] = {}
    # Comparable = la producen el lexico Y el rediseno. Se pregunta por los
    # metodos concretos y no por cuantos hay: con la via traducida una
    # dimension puede tener tres y el conteo la dejaria fuera en silencio.
    comparables = [dim for dim, sp in ESCALAS.items()
                   if "actual" in sp["metodos"] and "nuevo" in sp["metodos"]
                   and dim != "overall_sentiment"]
    for dim in comparables:
        sub = LARGO[LARGO.dimension == dim]
        # El pivote va solo sobre texto_id: pivot_table descarta los grupos cuya
        # clave es NaN, y con textos propios `lang` viene vacio.
        attrs = sub[["texto_id", "lang", "texto"]].drop_duplicates("texto_id")
        piv = (sub.pivot_table(index="texto_id", columns="metodo",
                               values="valencia", aggfunc="first")
                  .reset_index()
                  .merge(attrs, on="texto_id", how="left"))
        if "actual" not in piv or "nuevo" not in piv:
            continue
        d["valores"][dim] = {
            "puntos": [{"x": round(float(r["actual"]), 4),
                        "y": round(float(r["nuevo"]), 4),
                        "lang": r["lang"], "texto": r["texto"][:120]}
                       for _, r in piv.iterrows()],
            "escalonada": dim in ESCALONADAS,
            "marcas": ESCALONADAS.get(dim, []),
            "formula": ("clamp(mg * 0.8), con mg = clamp((pos - neg) * 0.2). "
                        "No incorpora el sentimiento: solo puede tomar {} valores."
                        ).format(len(ESCALONADAS[dim])) if dim in ESCALONADAS else
                       ("incorpora el sentimiento, que es continuo: no es escalonada, "
                        "es una nube desplazada a saltos"),
        }

    # ---- F4 · distribucion de relevancia y ausencia ------------------------
    d["ausencia"] = {}
    bordes = [i / 10 for i in range(11)]
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & (LARGO.metodo == "nuevo")]
        vals = sub["relevancia"].dropna().tolist()
        if not vals:
            continue
        conteo = [0] * 10
        for v in vals:
            idx = min(int(v * 10), 9)
            conteo[idx] += 1
        prop_muda = sum(1 for v in vals if v < UMBRAL_MUDEZ) / len(vals)
        d["ausencia"][dim] = {
            "bordes": bordes, "conteo": conteo, "n": len(vals),
            "prop_muda": round(prop_muda, 4),
        }
        if prop_muda > UMBRAL_DIMENSION_MUDA:
            _alerta("[{}] DIMENSION MAYORITARIAMENTE MUDA ({:.0%} bajo relevancia {}): "
                    "apenas mide nada en este corpus. Su promedio esta dominado por "
                    "textos que no hablaron del tema."
                    .format(dim, prop_muda, UMBRAL_MUDEZ))

    # lectura del control mudo, que es la prueba del eje de relevancia
    d["control_mudo"] = None
    if ES_BATERIA:
        cm = LARGO[(LARGO.frase == "control_mudo") & (LARGO.metodo == "nuevo")
                   & LARGO.dimension.isin(DIMS_CON_RELEVANCIA)]
        if not cm.empty:
            d["control_mudo"] = [
                {"dimension": dim,
                 "relevancia_media": round(float(g["relevancia"].mean()), 4),
                 "valencia_media": round(float(g["valencia"].mean()), 4)}
                for dim, g in cm.groupby("dimension")
            ]

    # ---- F5 · descomposicion de probabilidades ----------------------------
    d["probs"] = {}
    for grupo, g in PROBS.groupby("grupo"):
        etiquetas = g["etiqueta"].unique().tolist()
        clave = "frase" if ES_BATERIA and g["frase"].notna().all() else "texto_id"
        medias = g.pivot_table(index=clave, columns="etiqueta",
                               values="probabilidad", aggfunc="mean")
        d["probs"][grupo] = {
            "etiquetas": etiquetas,
            "filas": medias.index.tolist(),
            "valores": {e: [round(float(medias.loc[i, e]), 4) for i in medias.index]
                        for e in etiquetas if e in medias.columns},
            "agregado_por": clave,
        }
        # la alerta se calcula por observacion, no sobre las medias
        na_lab = "na" if "na" in etiquetas else ("neutral" if "neutral" in etiquetas else None)
        if na_lab:
            por_texto = g.pivot_table(index="texto_id", columns="etiqueta",
                                      values="probabilidad", aggfunc="first")
            if na_lab in por_texto.columns:
                domina = (por_texto[na_lab] > 0.5).mean()
                if domina > 0.8:
                    _alerta("[{}] LA HIPOTESIS DE IRRELEVANCIA DOMINA en {:.0%} de las "
                            "observaciones. Indicio de que las frases de este grupo "
                            "estan mal calibradas, no necesariamente de que nadie "
                            "hable del tema.".format(grupo, domina))

    # ---- F6 · cobertura del lexico por idioma -----------------------------
    # Se calcula siempre. Sin idiomas declarados el desglose cae en un solo
    # cubo: la cobertura y, sobre todo, el emparejamiento por subcadena siguen
    # siendo medibles, y es justo con tuits reales cuando mas importan.
    d["lexico"] = None
    if True:
        por_idioma = bool(LARGO["lang_grupo"].notna().any())
        textos = LARGO[["texto_id", "lang_grupo"]].drop_duplicates()
        textos = textos.rename(columns={"lang_grupo": "lang"})
        textos = textos.assign(
            lang=textos["lang"].fillna("sin idioma declarado")
            if not por_idioma else textos["lang"])
        filas = []
        for lang, g in textos.groupby("lang"):
            ids = set(g["texto_id"])
            sub = LEXICO[LEXICO.texto_id.isin(ids)] if len(LEXICO) else LEXICO
            sub = sub[sub.metodo == "actual"] if len(sub) else sub
            con_match = sub["texto_id"].nunique() if len(sub) else 0
            filas.append({
                "lang": lang, "n_textos": len(ids),
                "cobertura": round(con_match / len(ids), 4) if len(ids) else 0.0,
                "matches": int(len(sub)),
                "subcadenas": int(sub["es_subcadena"].sum()) if len(sub) else 0,
            })
        total_m = sum(f["matches"] for f in filas)
        total_s = sum(f["subcadenas"] for f in filas)
        ejemplos = []
        if len(LEXICO):
            for _, r in LEXICO[LEXICO.es_subcadena].head(8).iterrows():
                t = LARGO[LARGO.texto_id == r["texto_id"]]["texto"]
                ejemplos.append({"palabra": r["palabra"], "dimension": r["dimension"],
                                 "lang": r["lang"],
                                 "texto": (t.iloc[0][:110] if len(t) else "")})
        d["lexico"] = {"filas": filas, "total_matches": total_m,
                       "total_subcadenas": total_s, "ejemplos": ejemplos,
                       "medido": total_m > 0, "por_idioma": por_idioma}
        sin_cobertura = [f["lang"] for f in filas if f["cobertura"] == 0]
        if sin_cobertura and por_idioma:
            _alerta("COBERTURA LEXICA NULA en {}: en esos idiomas el lexico no activa "
                    "ninguna palabra, asi que sus {} dimensiones lexicas valen lo "
                    "mismo por construccion. No es una medicion, es el instrumento."
                    .format(", ".join(sin_cobertura), len(SETS_LEXICO)))

    # ---- estratos de recoleccion ------------------------------------------
    # Un promedio sobre un corpus dirigido por consulta mide sobre todo la
    # consulta. Se desglosa, y se avisa cuando el agregado mezcla estratos que
    # buscaban expresamente la dimension que se esta promediando.
    d["estratos"] = None
    _tiene_estrato = len(LARGO) and LARGO["estrato"].notna().any()
    if _tiene_estrato:
        _textos_estrato = LARGO[["texto_id", "estrato"]].drop_duplicates()
        _n_por_estrato = _textos_estrato["estrato"].value_counts().to_dict()
        _total = sum(_n_por_estrato.values())
        filas_e = []
        _nuevo = LARGO[LARGO.metodo == "nuevo"]
        for est, n_e in sorted(_n_por_estrato.items(), key=lambda p: -p[1]):
            g = _nuevo[_nuevo.estrato == est]
            por_dim = {}
            for dim, gd in g.groupby("dimension"):
                if dim == "overall_sentiment":
                    continue
                por_dim[dim] = {
                    "valencia": round(float(gd["valencia"].mean()), 4),
                    "relevancia": (round(float(gd["relevancia"].mean()), 4)
                                   if gd["relevancia"].notna().any() else None),
                    "n": int(gd["texto_id"].nunique()),
                }
            filas_e.append({"estrato": est, "n": int(n_e),
                            "proporcion": round(n_e / _total, 4),
                            "dimensiones": por_dim})
        d["estratos"] = {"filas": filas_e, "n_estratos": len(filas_e),
                         "dirigidas": CONSULTAS_DIRIGIDAS}

        for dim, claves in CONSULTAS_DIRIGIDAS.items():
            if dim not in set(LARGO["dimension"]):
                continue
            dirigidos = [e for e in _n_por_estrato
                         if any(k in str(e).lower() for k in claves)]
            if not dirigidos:
                continue
            prop = sum(_n_por_estrato[e] for e in dirigidos) / _total
            if 0 < prop < 1:
                _alerta("[{}] AGREGADO DIRIGIDO POR LA CONSULTA: el {:.0%} del "
                        "corpus viene de consultas que buscaban expresamente esta "
                        "dimension ({}). Su promedio sobre el corpus completo mide "
                        "la consulta tanto como el fenomeno, y NO es reportable "
                        "como hallazgo sobre la poblacion. Lee el desglose por "
                        "estrato.".format(dim, prop, ", ".join(sorted(dirigidos))))

    d["alertas"] = _ALERTAS
    return d


DATOS_FIGURAS = preparar_datos()
_payload = json.dumps(DATOS_FIGURAS, ensure_ascii=False)
print("conjunto embebido: {:.1f} KB".format(len(_payload.encode("utf-8")) / 1024))

In [ ]:
PLANTILLA_HTML = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Figuras sobre las dimensiones</title>
<script src="__CDN__"></script>
<style>
  :root{
    color-scheme: light;
    --sup:#fcfcfb; --plano:#f9f9f7; --tinta:#0b0b0b; --tinta2:#52514e;
    --mudo:#898781; --grid:#e1e0d9; --eje:#c3c2b7; --alerta:#d03b3b; --ok:#0ca30c;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--plano);color:var(--tinta);
       font:14px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1080px;margin:0 auto;padding-block:28px;padding-left:16px;padding-right:16px}
  h1{font-size:22px;margin:0 0 4px}
  h2{font-size:16px;margin:34px 0 10px;padding-bottom:6px;border-bottom:1px solid var(--grid)}
  h3{font-size:15px;margin:0 0 2px}
  p{margin:6px 0}
  .sub{color:var(--tinta2);margin:0 0 18px}
  .fig{background:var(--sup);border:1px solid var(--grid);border-radius:8px;
       padding:16px;margin:14px 0}
  .preg{color:var(--tinta2);font-style:italic;margin:0 0 8px}
  .n{display:inline-block;font-size:11px;color:var(--tinta2);background:var(--plano);
     border:1px solid var(--grid);border-radius:99px;padding:1px 9px;margin:0 0 8px;
     font-variant-numeric:tabular-nums}
  .aviso{background:#fff6e8;border-left:3px solid #eda100;padding:8px 11px;
         font-size:13px;color:var(--tinta2);margin:8px 0}
  .nota{font-size:13px;color:var(--tinta2);margin-top:10px}
  .cifra{font-variant-numeric:tabular-nums;font-weight:600;color:var(--tinta)}
  .lienzo{width:100%}
  .banner{background:#fff6e8;border:1px solid #eda100;border-radius:8px;
          padding:12px 14px;margin:0 0 16px}
  .banner b{color:#8a5a00}
  .alertas{background:var(--sup);border:1px solid var(--alerta);border-radius:8px;
           padding:12px 14px;margin:14px 0}
  .alertas h3{color:var(--alerta)}
  .alertas li{margin:6px 0;color:var(--tinta2)}
  .bien{background:var(--sup);border:1px solid var(--grid);border-left:3px solid var(--ok);
        border-radius:8px;padding:10px 14px;margin:14px 0;color:var(--tinta2)}
  details{margin-top:10px}
  summary{cursor:pointer;font-size:13px;color:var(--tinta2)}
  table{border-collapse:collapse;width:100%;font-size:12px;margin-top:8px;
        font-variant-numeric:tabular-nums}
  th,td{border-bottom:1px solid var(--grid);padding:4px 7px;text-align:left;vertical-align:top}
  th{color:var(--tinta2);font-weight:600}
  td.num{text-align:right}
  .tabla-scroll{overflow-x:auto}
  .fallo{background:var(--sup);border:1px solid var(--alerta);border-radius:8px;padding:20px}
  code{background:var(--plano);border:1px solid var(--grid);border-radius:3px;padding:0 4px;
       font-size:12px}
  .rtl{direction:rtl;text-align:right;unicode-bidi:isolate}
</style>
</head>
<body>
<div class="wrap" id="raiz"><p>cargando…</p></div>
<script>
const DATOS = __DATOS__;
const raiz = document.getElementById('raiz');

function el(h){const t=document.createElement('template');t.innerHTML=h.trim();return t.content.firstChild;}
function add(h){raiz.appendChild(el(h));}
function esc(s){return String(s==null?'':s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));}
function rtl(lang,s){return lang==='ar' ? '<span class="rtl">'+esc(s)+'</span>' : esc(s);}
function fix(v,n){return (v==null||isNaN(v))?'—':Number(v).toFixed(n==null?3:n);}

const P = DATOS.paleta;
/* Los tres métodos. 'traducido' es el mismo léxico sobre la columna de
   traducción; se distingue por trama además de por tono, para que siga
   siendo legible impreso en escala de grises. */
const NOMBRES_METODO = {actual:'léxico (original)', traducido:'léxico (traducido)',
                        nuevo:'rediseño zero-shot'};
function NOMBRE_METODO(m){return NOMBRES_METODO[m]||m;}
function COLOR_METODO(m){return m==='nuevo'?P.nuevo:(m==='traducido'?(P.traducido||P.mudo):P.actual);}
const TOOLBOX = {right:6,top:-2,feature:{saveAsImage:{type:'svg',title:'guardar SVG',
  name:'figura',backgroundColor:'#fcfcfb'}}};
const EJE_X = {axisLine:{lineStyle:{color:P.eje}},axisTick:{show:false},
  axisLabel:{color:P.tinta2,fontSize:11}};
const EJE_Y = {axisLine:{show:false},axisTick:{show:false},
  axisLabel:{color:P.tinta2,fontSize:11},
  splitLine:{lineStyle:{color:P.grid,width:1}}};

function init(id,opt){
  const dom=document.getElementById(id);
  const c=echarts.init(dom,null,{renderer:'svg'});
  c.setOption(opt);
  window.addEventListener('resize',()=>c.resize());
  return c;
}

function bloque(id,titulo,pregunta,n,nota,tablaHTML,alto){
  const bajo = (n!==null && n!==undefined && n < DATOS.n_minimo);
  add('<section class="fig">'
    + '<h3>'+titulo+'</h3>'
    + '<p class="preg">'+pregunta+'</p>'
    + ((n!==null&&n!==undefined)?'<p class="n">n = '+n+'</p>':'')
    + (bajo?'<p class="aviso"><b>n = '+n+', por debajo de '+DATOS.n_minimo+'.</b> '
        +'A esta escala la figura no es interpretable. Se dibuja para inspeccionar el '
        +'instrumento, no para reportar un resultado.</p>':'')
    + '<div id="'+id+'" class="lienzo" style="height:'+(alto||340)+'px"></div>'
    + (nota?'<p class="nota">'+nota+'</p>':'')
    + (tablaHTML?'<details><summary>ver los datos de esta figura</summary>'
        +'<div class="tabla-scroll">'+tablaHTML+'</div></details>':'')
    + '</section>');
}

function tabla(cols,filas){
  let h='<table><thead><tr>';
  cols.forEach(c=>h+='<th>'+esc(c)+'</th>');
  h+='</tr></thead><tbody>';
  filas.forEach(f=>{h+='<tr>';f.forEach(v=>{
    const num = (typeof v==='number');
    h+='<td'+(num?' class="num"':'')+'>'+(num?fix(v):v)+'</td>';});h+='</tr>';});
  return h+'</tbody></table>';
}

function dibujar(){
  raiz.innerHTML='';
  add('<h1>Figuras sobre las dimensiones</h1>'
    + '<p class="sub">Comparación del método léxico actual contra el rediseño zero-shot. '
    + DATOS.n_textos+' textos · '+DATOS.ficha.pasadas_por_texto+' pasadas de NLI por texto.</p>');

  /* Rótulo de procedencia: TRES estados, no dos. El del corpus no dice
     "no reportable" sino algo distinto y más fácil de pasar por alto:
     descriptivo pero no validado. */
  if(DATOS.procedencia==='bateria'){
    add('<div class="banner"><b>BATERÍA DE PRUEBA DEL INSTRUMENTO — no es una muestra.</b>'
      + '<p>Estas figuras se construyeron sobre cinco frases escritas a propósito, cada una '
      + 'en seis idiomas. Sirven para ver cómo se comporta el instrumento cuando el contenido '
      + 'no varía. <b>Ningún número de esta página es reportable como hallazgo sobre el corpus.</b></p>'
      + '<p>Traducciones: '+esc(DATOS.ficha.traduccion||'—')+'. Si una versión resulta más '
      + 'enfática que otra, parte de la dispersión atribuida al instrumento es artefacto de '
      + 'traducción; están abajo, completas, para que se puedan auditar.</p></div>');
  } else if(DATOS.procedencia==='corpus'){
    add('<div class="banner"><b>CORPUS REAL — descriptivo, NO validado.</b>'
      + '<p>'+DATOS.n_textos+' textos del corpus del proyecto. A diferencia de la batería, '
      + 'estos números <b>sí</b> hablan del corpus — pero <b>ninguno está validado</b>: el '
      + 'archivo no trae codificación humana, así que el acuerdo de cada método contra '
      + 'codificadores sigue sin medirse.</p>'
      + '<p><b>Es el riesgo de leer esta página demasiado rápido:</b> un corpus de este tamaño '
      + 'invita a tratar sus promedios como hallazgos firmes cuando todavía no lo son.</p></div>');
  } else {
    add('<div class="banner"><b>Textos aportados a mano.</b>'
      + '<p>Ni batería ni corpus: lo que digan estas figuras vale para estos '
      + DATOS.n_textos+' textos y nada más.</p></div>');
  }

  if(DATOS.alertas && DATOS.alertas.length){
    let h='<div class="alertas"><h3>Alertas</h3><ul>';
    DATOS.alertas.forEach(a=>h+='<li>'+esc(a)+'</li>');
    add(h+'</ul></div>');
  } else {
    add('<div class="bien">Ninguna dimensión superó los umbrales configurados '
      + '(mudez '+DATOS.umbral_mudez+', dimensión muda '
      + Math.round(DATOS.umbral_dim_muda*100)+'%).</div>');
  }

  add('<div class="bien" style="border-left-color:var(--eje)"><b>Estas figuras describen; no validan.</b> '
    + 'Muestran cómo se comporta cada instrumento, no cuál mide mejor la realidad. Para eso hace '
    + 'falta codificar a mano una muestra (~200 tuits) y calcular el alfa de Krippendorff de cada '
    + 'método contra esos códigos, con umbral 0.67, como indica la última sección del notebook.</div>');

  seccionDispersion();
  seccionPlano();
  seccionValores();
  seccionAusencia();
  seccionProbs();
  seccionLexico();
  seccionEstratos();
  seccionAncla();
  seccionFicha();
}

/* ---------- verificación del ancla geográfica ---------- */
function seccionAncla(){
  add('<h2>8 · ¿Importa dónde vive el ancla «México»?</h2>');
  const A=DATOS.ancla;
  if(!A){
    add('<div class="aviso"><b>Sin medir.</b> El ancla geográfica vive en la '
      + (DATOS.ancla_en_etiquetas?'<b>etiquetas</b>':'<b>plantilla</b>')
      + ', y esa decisión se apoya <b>solo en el argumento mecánico</b> de la sección 2: '
      + 'el softmax cancela un prefijo constante, no un predicado que encarece las '
      + 'afirmaciones y abarata la negación. Para respaldarla —o refutarla— pon '
      + '<code>COMPARAR_ANCLA = True</code> y vuelve a correr la batería.</div>');
    return;
  }
  const filas=A.filas.map(f=>[f.dimension,
    fix(f.plantilla.relevancia_media,4), fix(f.etiquetas.relevancia_media,4),
    fix(f.delta_relevancia,4),
    Math.round(f.plantilla.prop_muda*100)+'% / '+Math.round(f.etiquetas.prop_muda*100)+'%']);
  const veredicto = A.prediccion_confirmada
    ? '<b>Predicción confirmada.</b> El ancla dentro de las etiquetas baja la relevancia '
      + '(delta medio <span class="cifra">'+fix(A.delta_medio,4)+'</span>), que es lo que '
      + 'la sección 2 argumenta: infla <code>na</code> porque añadir el predicado a una '
      + 'negación la hace más fácil de entrañar.'
    : '<b>Predicción NO confirmada.</b> El ancla dentro de las etiquetas <b>no</b> baja la '
      + 'relevancia aquí (delta medio <span class="cifra">'+fix(A.delta_medio,4)+'</span>). '
      + 'El argumento mecánico de la sección 2 no se sostiene tal como está escrito, y la '
      + 'decisión de diseño hay que revisarla antes de reportar nada.';
  const id='f8';
  bloque(id,'relevancia media según dónde viva el ancla',
    '¿La redacción de las hipótesis, y no el corpus, es lo que hunde la relevancia?',
    A.filas.length, veredicto,
    tabla(['dimensión','ancla en plantilla','ancla en etiquetas','delta','% mudo (plant/etiq)'],filas),
    300);
  init(id,{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
    legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22},
    toolbox:TOOLBOX,
    grid:{left:52,right:20,top:18,bottom:52},
    xAxis:Object.assign({type:'category',data:A.filas.map(f=>f.dimension)},EJE_X),
    yAxis:Object.assign({type:'value',min:0,max:1,name:'relevancia media'},EJE_Y),
    series:[
      {name:'ancla en la plantilla',type:'bar',barGap:'12%',
       data:A.filas.map(f=>f.plantilla.relevancia_media),
       itemStyle:{color:P.nuevo,borderRadius:3},
       label:{show:true,position:'top',fontSize:10,color:P.tinta2,
              formatter:p=>Number(p.value).toFixed(2)}},
      {name:'ancla en las etiquetas',type:'bar',
       data:A.filas.map(f=>f.etiquetas.relevancia_media),
       itemStyle:{color:P.mudo,borderRadius:3,
         decal:{symbol:'line',dashArrayX:[1,0],dashArrayY:[2,4],
                rotation:-Math.PI/4,color:'rgba(0,0,0,.22)'}},
       label:{show:true,position:'top',fontSize:10,color:P.tinta2,
              formatter:p=>Number(p.value).toFixed(2)}}
    ]
  });
  add('<div class="nota">Umbral de mudez usado: <span class="cifra">'+A.umbral_mudez
    + '</span>. Dimensiones declaradas mudas — con el ancla en la plantilla: <span class="cifra">'
    + A.dimensiones_mudas.plantilla+'</span>; en las etiquetas: <span class="cifra">'
    + A.dimensiones_mudas.etiquetas+'</span>.</div>');
}

/* ---------- estratos de recolección ---------- */
function seccionEstratos(){
  const E=DATOS.estratos;
  if(!E || !E.filas || !E.filas.length) return;
  add('<h2>7 · ¿De qué consulta viene cada número?</h2>');
  add('<p class="sub">El corpus no es una masa homogénea: se recolectó con consultas distintas, '
    + 'y algunas buscaban expresamente una de las dimensiones que aquí se promedian. Promediar '
    + 'sobre el conjunto mide la consulta tanto como el fenómeno. Por eso se desglosa.</p>');
  const dims=[...new Set(E.filas.flatMap(f=>Object.keys(f.dimensiones)))].sort();
  const filas=E.filas.map(f=>[f.estrato, f.n, Math.round(f.proporcion*100)+'%']
    .concat(dims.map(d=>f.dimensiones[d]?fix(f.dimensiones[d].valencia,3):'—')));
  const id='f7';
  bloque(id,'valencia media por estrato',
    '¿El promedio cambia según con qué consulta se recolectó el texto?',
    E.filas.reduce((a,f)=>a+f.n,0),
    '<b>Un promedio que mezcla estratos no es reportable como hallazgo sobre la población.</b> '
    + 'Las alertas de arriba dicen cuáles están dirigidos.',
    tabla(['estrato','n','% del corpus'].concat(dims),filas),
    Math.max(220, 60+E.filas.length*28));
  init(id,{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
    legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:18,type:'scroll',width:'92%'},
    toolbox:TOOLBOX,
    grid:{left:170,right:24,top:14,bottom:52},
    xAxis:Object.assign({type:'value',min:-1,max:1},EJE_X),
    yAxis:Object.assign({type:'category',
      data:E.filas.map(f=>String(f.estrato).slice(0,26))},EJE_Y),
    series:dims.map((d,k)=>({
      name:d,type:'bar',
      data:E.filas.map(f=>f.dimensiones[d]?f.dimensiones[d].valencia:null),
      itemStyle:{borderRadius:2},
      label:{show:false}}))
  });
}

/* ---------- F1 · dispersión entre idiomas ---------- */
function seccionDispersion(){
  const D=DATOS.dispersion; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>1 · ¿Mide lo mismo cuando el contenido es el mismo?</h2>');
  add('<p class="sub">El mismo contenido, en varios idiomas. Un instrumento válido debería darles '
    + 'aproximadamente el mismo valor. La dispersión —rango y desviación— es el número que se '
    + 'reporta: mide cuánto varía el instrumento cuando el contenido no varía. Idealmente, cero. '
    + 'Los idiomas salen de los datos cargados, no de una lista declarada.</p>');
  if(DATOS.idiomas_escasos && Object.keys(DATOS.idiomas_escasos).length){
    add('<div class="aviso">Excluidos por tener menos de '+DATOS.n_minimo_idioma
      + ' observaciones: '
      + Object.entries(DATOS.idiomas_escasos).map(([l,n])=>esc(l)+' (n='+n+')').join(', ')
      + '. Se nombran para que la exclusión se vea, en vez de dibujarlos como comparables.</div>');
  }
  if(DATOS.dispersion_excluidas && DATOS.dispersion_excluidas.length){
    let h='<div class="aviso"><b>Dimensiones que no pueden aparecer aquí:</b><ul>';
    DATOS.dispersion_excluidas.forEach(x=>h+='<li><code>'+esc(x.dimension)+'</code> — '+esc(x.razon)+'</li>');
    add(h+'</ul></div>');
  }
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f1_'+i;
    const metodos = v.metodos || ['actual','nuevo'];
    let nota='<b>Dispersión entre idiomas.</b> ';
    metodos.forEach(m=>{const st=v.stats[m];
      nota += NOMBRE_METODO(m)+': rango <span class="cifra">'+fix(st&&st.rango,4)+'</span>, '
            + 'σ <span class="cifra">'+fix(st&&st.sigma,4)+'</span>. ';});
    if(metodos.indexOf('traducido')>=0){
      nota += '<br><b>Cuidado con «léxico traducido»:</b> es el MISMO léxico corriendo sobre la '
            + 'columna de traducción, así que mide el léxico junto con la calidad de una '
            + 'traducción automática. Si gana, no se sabe cuánto mérito es de cada uno.';
    }
    const filas=v.langs.map((l,k)=>[l].concat(metodos.map(m=>v.series[m][k])));
    bloque(id,'<code>'+dim+'</code> por idioma',
      '¿Cuánto cambia la medición cuando solo cambia el idioma?',
      v.langs.length,nota,
      tabla(['idioma'].concat(metodos.map(NOMBRE_METODO)),filas),320);
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:52,right:20,top:18,bottom:52},
      xAxis:Object.assign({type:'category',data:v.langs},EJE_X),
      yAxis:Object.assign({type:'value',min:-1,max:1},EJE_Y),
      series:metodos.map((m,k)=>({
        name:NOMBRE_METODO(m),type:'bar',data:v.series[m],
        barGap:k===0?'12%':undefined,
        itemStyle:{color:COLOR_METODO(m),borderRadius:3,
                   decal:m==='traducido'?{symbol:'line',dashArrayX:[1,0],
                     dashArrayY:[2,4],rotation:-Math.PI/4,color:'rgba(0,0,0,.22)'}:null},
        label:{show:true,position:'top',fontSize:10,color:P.tinta2,
               formatter:p=>Number(p.value).toFixed(2)}}))
    });
  });
}

/* ---------- F2 · plano valencia × relevancia ---------- */
const SIMBOLOS=['circle','diamond','triangle','rect','pin'];
function seccionPlano(){
  const D=DATOS.plano; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>2 · ¿Cuántos de los ceros de hoy son «no habló»?</h2>');
  add('<p class="sub">Dos ejes que el método actual colapsa en uno. Un valor cercano a cero '
    + 'arriba a la derecha significa «habló del tema y su juicio fue neutro»; el mismo valor '
    + 'a la izquierda significa «no habló». Hoy ambos son el mismo <code>0.0000</code> y se '
    + 'promedian juntos. Las formas distinguen la frase de origen, para que la figura se lea '
    + 'también impresa en blanco y negro.</p>');
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f2_'+i;
    const grupos=Object.keys(v.grupos);
    const series=grupos.map((g,k)=>({
      name:g,type:'scatter',symbol:SIMBOLOS[k%SIMBOLOS.length],symbolSize:12,
      itemStyle:{color:P.actual,borderColor:P.sup,borderWidth:2},
      data:v.grupos[g].map(p=>[p.x,p.y,p.lang,p.texto])
    }));
    series[0].markArea={silent:true,itemStyle:{color:'rgba(137,135,129,0.13)'},
      label:{show:true,position:'insideTop',color:P.mudo,fontSize:11,
             formatter:'no habló del tema'},
      data:[[{xAxis:0},{xAxis:DATOS.umbral_mudez}]]};
    series[0].markLine={silent:true,symbol:'none',
      lineStyle:{color:P.eje,type:'dashed'},label:{show:false},
      data:[{yAxis:0}]};
    const filas=[];grupos.forEach(g=>v.grupos[g].forEach(p=>filas.push(
      [g,p.lang,p.x,p.y,p.texto])));
    const nota='<b>'+v.mudos+' de '+v.n+'</b> observaciones caen en la banda de mudez '
      + '(relevancia &lt; '+DATOS.umbral_mudez+'). No se filtran: dibujarlas <em>es</em> la '
      + 'evidencia. El umbral es un corte de lectura, no un filtro de datos.';
    bloque(id,'<code>'+dim+'</code> — valencia contra relevancia',
      '¿La observación no opinó, o no habló del tema?',
      v.n,nota,tabla(['frase','idioma','relevancia','valencia','texto'],filas),360);
    init(id,{
      tooltip:{trigger:'item',formatter:p=>
        '<b>'+esc(p.seriesName)+'</b> · '+esc(p.data[2]||'')+'<br>'
        +'relevancia '+fix(p.data[0],2)+' · valencia '+fix(p.data[1],2)
        +'<br><span style="color:#52514e">'+esc((p.data[3]||'').slice(0,70))+'…</span>'},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:60,right:24,top:18,bottom:78},
      xAxis:Object.assign({type:'value',min:0,max:1,name:'relevancia',
        nameLocation:'middle',nameGap:30,nameTextStyle:{color:P.tinta2}},EJE_X,
        {splitLine:{lineStyle:{color:P.grid}}}),
      yAxis:Object.assign({type:'value',min:-1,max:1,name:'valencia',
        nameLocation:'middle',nameGap:38,nameTextStyle:{color:P.tinta2}},EJE_Y),
      series:series
    });
  });
  if(DATOS.plano_excluidas && DATOS.plano_excluidas.length){
    let h='<p class="nota"><b>Dimensiones excluidas de esta figura:</b> ';
    h+=DATOS.plano_excluidas.map(e=>'<code>'+esc(e.dimension)+'</code> — '+esc(e.razon)).join('; ');
    add(h+'.</p>');
  }
  if(DATOS.control_mudo){
    let h='<div class="bien"><b>El control mudo.</b> El texto sobre comida no habla de '
      + 'hospitalidad, ambiente ni autoridades. Si el eje de relevancia funciona, debe salir '
      + 'bajo en las tres:<ul>';
    DATOS.control_mudo.forEach(c=>{var bajo=c.relevancia_media<DATOS.umbral_mudez;
      h+='<li><code>'+esc(c.dimension)+'</code>: relevancia media '
      + '<span class="cifra">'+fix(c.relevancia_media,2)+'</span> '
      + (bajo?'<b style="color:var(--ok)">por debajo del umbral, como debe ser</b>'
            :'<b style="color:var(--alerta)">POR ENCIMA del umbral '+DATOS.umbral_mudez
              +': el modelo cree que este texto sí habla del tema</b>')
      + ', valencia media <span class="cifra">'+fix(c.valencia_media,2)+'</span></li>';});
    add(h+'</ul>El método actual asigna a este mismo texto un 0.0000 indistinguible del de un '
      + 'turista que habló del tema y lo juzgó neutro.</div>');
  }
}

/* ---------- F3 · valores posibles de cada método ---------- */
function seccionValores(){
  const D=DATOS.valores; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>3 · ¿Qué valores puede tomar cada método?</h2>');
  add('<p class="sub">Cada punto es un texto. La diagonal es el acuerdo perfecto entre ambos '
    + 'métodos: cuanto más lejos de ella, más discrepan.</p>');
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f3_'+i;
    const series=[
      {name:'acuerdo perfecto',type:'line',silent:true,data:[[-1,-1],[1,1]],symbol:'none',silent:true,
       lineStyle:{color:P.eje,type:'dashed',width:1},tooltip:{show:false}},
      {name:'textos',type:'scatter',symbolSize:12,
       itemStyle:{color:P.nuevo,borderColor:P.sup,borderWidth:2},
       data:v.puntos.map(p=>[p.x,p.y,p.lang,p.texto])}
    ];
    if(v.escalonada && v.marcas.length){
      series[1].markLine={silent:true,symbol:'none',
        lineStyle:{color:P.actual,type:'dotted',width:1,opacity:0.55},
        label:{show:false},
        data:v.marcas.map(m=>({xAxis:m}))};
    }
    const nota = v.escalonada
      ? '<b>Las líneas verticales son los únicos valores que el método actual puede tomar aquí</b> '
        + '('+v.marcas.length+' en total). Fórmula: '+esc(v.formula)
      : '<b>Esta dimensión no es escalonada.</b> Su fórmula actual '+esc(v.formula)
        + ', así que no tiene sentido marcar valores discretos: no existen.';
    bloque(id,'<code>'+dim+'</code> — actual contra rediseño',
      '¿Dónde discrepan los dos métodos, y qué valores puede producir cada uno?',
      v.puntos.length,nota,
      tabla(['idioma','actual','rediseño','texto'],
            v.puntos.map(p=>[p.lang,p.x,p.y,p.texto])),360);
    init(id,{
      tooltip:{trigger:'item',formatter:p=>p.seriesName==='textos'
        ? esc(p.data[2]||'')+'<br>actual '+fix(p.data[0],3)+' · rediseño '+fix(p.data[1],3)
          +'<br><span style="color:#52514e">'+esc((p.data[3]||'').slice(0,70))+'…</span>'
        : ''},
      toolbox:TOOLBOX,
      grid:{left:60,right:24,top:18,bottom:56},
      xAxis:Object.assign({type:'value',min:-1,max:1,name:'método actual',
        nameLocation:'middle',nameGap:30,nameTextStyle:{color:P.tinta2}},EJE_X,
        {splitLine:{lineStyle:{color:P.grid}}}),
      yAxis:Object.assign({type:'value',min:-1,max:1,name:'rediseño',
        nameLocation:'middle',nameGap:38,nameTextStyle:{color:P.tinta2}},EJE_Y),
      series:series
    });
  });
}

/* ---------- F4 · ausencia por dimensión ---------- */
function seccionAusencia(){
  const D=DATOS.ausencia; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>4 · ¿Esta dimensión mide algo en este corpus?</h2>');
  add('<p class="sub">Distribución <em>completa</em> de la relevancia, con el umbral superpuesto. '
    + 'Se muestra entera a propósito: el corte en '+DATOS.umbral_mudez+' es arbitrario y hay que '
    + 'poder juzgar qué tan sensible es la lectura al valor elegido.</p>');
  const etq=[];for(let k=0;k<10;k++)etq.push((k/10).toFixed(1)+'–'+((k+1)/10).toFixed(1));
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f4_'+i;
    const corte=Math.round(DATOS.umbral_mudez*10);
    const muda = v.prop_muda > DATOS.umbral_dim_muda;
    const nota='<b>'+Math.round(v.prop_muda*100)+'%</b> de las observaciones quedan bajo el '
      + 'umbral de mudez. '+(muda
        ? '<span style="color:var(--alerta)">Supera el '
          + Math.round(DATOS.umbral_dim_muda*100)+'% configurado: en este corpus la dimensión '
          + 'apenas mide nada, y su promedio está dominado por textos que no hablaron del tema.</span>'
        : 'Por debajo del '+Math.round(DATOS.umbral_dim_muda*100)+'% configurado.')
      + ' La ausencia se reporta como magnitud propia, nunca apilada junto a la valencia.';
    bloque(id,'<code>'+dim+'</code> — distribución de relevancia',
      '¿Hablaron del tema, o la dimensión está muda?',
      v.n,nota,tabla(['rango de relevancia','observaciones'],
                     etq.map((e,k)=>[e,v.conteo[k]])),300);
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
      toolbox:TOOLBOX,
      grid:{left:52,right:20,top:18,bottom:40},
      xAxis:Object.assign({type:'category',data:etq},EJE_X),
      yAxis:Object.assign({type:'value',minInterval:1},EJE_Y),
      series:[{type:'bar',data:v.conteo,
        itemStyle:{color:p=>p.dataIndex<corte?P.mudo:P.nuevo,borderRadius:3},
        label:{show:true,position:'top',fontSize:10,color:P.tinta2,
               formatter:p=>p.value>0?p.value:''},
        markArea:{silent:true,itemStyle:{color:'rgba(137,135,129,0.13)'},
          label:{show:true,position:'insideTopLeft',color:P.mudo,fontSize:11,
                 formatter:'zona muda'},
          data:corte>0?[[{xAxis:etq[0]},{xAxis:etq[Math.max(corte-1,0)]}]]:[]}
      }]
    });
  });
}

/* ---------- F5 · descomposición de probabilidades ---------- */
const COLOR_ETIQUETA={pos:'#2a78d6',na:'#898781',neg:'#e34948',
  safety:'#2a78d6',neutral:'#898781',insecurity:'#eda100',violence:'#e34948'};
const ORDEN_ETIQUETA=['pos','safety','na','neutral','insecurity','neg','violence'];
function seccionProbs(){
  const D=DATOS.probs; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>5 · ¿A dónde se va la probabilidad del modelo?</h2>');
  add('<p class="sub">Las hipótesis de cada grupo compiten y suman 1. Si la hipótesis de '
    + 'irrelevancia se lo lleva casi todo en todas partes, las frases están mal calibradas: '
    + 'es un problema del instrumento, no del corpus. Las frases exactas están en la ficha '
    + 'técnica, al final.</p>');
  claves.forEach((grupo,i)=>{
    const v=D[grupo], id='f5_'+i;
    const etq=v.etiquetas.slice().sort((a,b)=>{
      const ia=ORDEN_ETIQUETA.indexOf(a),ib=ORDEN_ETIQUETA.indexOf(b);
      return (ia<0?99:ia)-(ib<0?99:ib);});
    const series=etq.filter(e=>v.valores[e]).map(e=>({
      name:e,type:'bar',stack:'p',
      data:v.valores[e],
      itemStyle:{color:COLOR_ETIQUETA[e]||P.mudo,borderColor:P.sup,borderWidth:2},
      label:{show:true,position:'inside',fontSize:10,color:'#fff',
             formatter:p=>p.value<=0.10?'':((p.dataIndex===0&&p.value>0.16)
               ? e+'  '+Number(p.value).toFixed(2)
               : Number(p.value).toFixed(2))}
    }));
    const filas=v.filas.map((f,k)=>[f].concat(etq.map(e=>v.valores[e]?v.valores[e][k]:null)));
    const nota='Promedio por <code>'+esc(v.agregado_por)+'</code>. Cada barra suma 1. '
      + 'Las etiquetas van en orden semántico: positiva, irrelevancia, negativa.';
    bloque(id,'grupo <code>'+grupo+'</code>',
      '¿Cómo reparte el modelo la probabilidad entre las hipótesis?',
      null,nota,tabla(['fila'].concat(etq),filas),
      Math.max(180,60+v.filas.length*42));
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'},
        valueFormatter:v2=>Number(v2).toFixed(3)},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:130,right:24,top:12,bottom:46},
      xAxis:Object.assign({type:'value',max:1},EJE_Y),
      yAxis:Object.assign({type:'category',data:v.filas},EJE_X,
        {axisLabel:{color:P.tinta2,fontSize:11}}),
      series:series
    });
  });
}

/* ---------- F6 · cobertura del léxico ---------- */
function seccionLexico(){
  const v=DATOS.lexico;
  if(!v) return;
  add('<h2>6 · ¿Llega a activarse el léxico del método actual?</h2>');
  add('<p class="sub">El método actual mide tres dimensiones contando palabras de listas en '
    + 'inglés y español. Donde no activa ninguna, las tres dimensiones valen lo mismo por '
    + 'construcción: no es una medición, es el instrumento.'
    + (v.por_idioma ? '' : ' <b>Los textos cargados no declaran idioma</b>, así que el '
        + 'desglose por idioma no aplica y todo cae en un solo grupo; el emparejamiento por '
        + 'subcadena se mide igual.')
    + '</p>');
  const langs=v.filas.map(f=>f.lang);
  const cob=v.filas.map(f=>f.cobertura);
  const nota_sub = v.total_matches===0
    ? '<b>No se midió en este corpus:</b> el léxico no activó ninguna palabra en ningún texto, '
      + 'así que no hay emparejamientos que clasificar. Eso <em>no</em> significa que el '
      + 'emparejamiento por subcadena no ocurra.'
    : '<b>'+v.total_subcadenas+' de '+v.total_matches+'</b> emparejamientos ocurrieron por '
      + 'subcadena y no por palabra completa. El método actual busca con <code>w in texto</code>, '
      + 'sin frontera de palabra: <code>safe</code> se activa dentro de <code>unsafe</code>, '
      + '<code>kind</code> dentro de <code>kinda</code>. Aquí solo se mide; corregirlo '
      + 'falsearía la comparación, porque dejaría de ser el método que está en producción.';
  let ej='';
  if(v.ejemplos && v.ejemplos.length){
    ej='<details><summary>emparejamientos por subcadena encontrados</summary>'
      + '<div class="tabla-scroll">'+tabla(['palabra','dimensión','idioma','texto'],
          v.ejemplos.map(e=>[e.palabra,e.dimension,e.lang,e.texto]))+'</div></details>';
  }
  bloque('f6','Cobertura del léxico por idioma',
    '¿En qué proporción de los textos de cada idioma el léxico activa alguna palabra?',
    v.filas.reduce((a,f)=>a+f.n_textos,0),
    nota_sub,
    tabla(['idioma','textos','cobertura','emparejamientos','por subcadena'],
      v.filas.map(f=>[f.lang,f.n_textos,f.cobertura,f.matches,f.subcadenas])),300);
  init('f6',{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'},
      valueFormatter:x=>Math.round(x*100)+'%'},
    toolbox:TOOLBOX,
    grid:{left:56,right:20,top:18,bottom:40},
    xAxis:Object.assign({type:'category',data:langs},EJE_X),
    yAxis:Object.assign({type:'value',min:0,max:1,
      axisLabel:{color:P.tinta2,fontSize:11,formatter:x=>Math.round(x*100)+'%'}},EJE_Y),
    series:[{type:'bar',data:cob,
      itemStyle:{color:p=>p.value===0?P.mudo:P.actual,borderRadius:3},
      label:{show:true,position:'top',fontSize:10,color:P.tinta2,
             formatter:p=>Math.round(p.value*100)+'%'}}]
  });
  if(ej) add(ej);
}

/* ---------- ficha técnica ---------- */
function seccionFicha(){
  const f=DATOS.ficha;
  add('<h2>Ficha técnica del instrumento</h2>');
  add('<p class="sub">Las hipótesis <em>son</em> el instrumento de medición: cambiarlas cambia '
    + 'los resultados. Van aquí textualmente para que la figura sea citable, igual que se cita '
    + 'un libro de códigos.</p>');
  let h='<section class="fig"><table><tbody>';
  h+='<tr><th>idioma de las hipótesis</th><td><code>'+esc(f.idioma_hipotesis)+'</code></td></tr>';
  h+='<tr><th>plantilla</th><td><code>'+esc(f.plantilla)+'</code></td></tr>';
  h+='<tr><th>modelo de sentimiento</th><td><code>'+esc(f.modelo_sentimiento)+'</code></td></tr>';
  h+='<tr><th>modelo NLI</th><td><code>'+esc(f.modelo_nli)+'</code></td></tr>';
  h+='<tr><th>pasadas de NLI por texto</th><td>'+f.pasadas_por_texto+'</td></tr>';
  if(f.tiempo_corrida_s!=null)
    h+='<tr><th>tiempo de la corrida</th><td>'+f.tiempo_corrida_s+' s</td></tr>';
  h+='<tr><th>fecha</th><td>'+esc(f.fecha)+'</td></tr>';
  h+='</tbody></table>';
  h+='<h3 style="margin-top:16px">Hipótesis</h3>'+tabla(['grupo','etiqueta','frase'],
     f.hipotesis.map(x=>[x.grupo,x.etiqueta,x.frase]));
  h+='</section>';
  add(h);

  if(DATOS.bateria && DATOS.bateria.length){
    let b='<section class="fig"><h3>Batería completa</h3>'
      + '<p class="preg">Publicada íntegra para que la equivalencia entre traducciones se pueda '
      + 'auditar. Si una versión no dice lo mismo que las otras, la figura 1 lo estará contando '
      + 'como sesgo del instrumento cuando es error de traducción.</p>'
      + '<div class="tabla-scroll"><table><thead><tr><th>frase</th><th>idioma</th>'
      + '<th>texto</th><th>qué prueba</th></tr></thead><tbody>';
    DATOS.bateria.forEach(t=>{b+='<tr><td>'+esc(t.frase)+'</td><td>'+esc(t.lang)+'</td><td>'
      + rtl(t.lang,t.texto)+'</td><td>'+esc(t.que_prueba)+'</td></tr>';});
    add(b+'</tbody></table></div></section>');
  }
}

/* ---------- arranque ----------
   Va al FINAL a proposito. Las declaraciones de funcion se elevan, pero
   `const P` y `const TOOLBOX` no: llamar a dibujar() antes de ellas las deja
   en zona muerta temporal y revienta con "Cannot access 'P' before
   initialization" en la primera figura, dejando la pagina a medio dibujar. */
if (typeof echarts === 'undefined') {
  raiz.innerHTML = '<div class="fallo"><h1>No se pudo cargar la biblioteca de gráficas</h1>'
    + '<p>Esta página dibuja con Apache ECharts, que se carga desde un CDN. La petición a '
    + '<code>__CDN__</code> no llegó.</p>'
    + '<p>Causas habituales: no hay conexión a internet, una red corporativa bloquea el CDN, '
    + 'o el archivo se abrió sin red. Los datos <em>sí</em> están dentro de este archivo: '
    + 'recupera la conexión y recarga, y las figuras aparecen sin volver a ejecutar el notebook.</p></div>';
} else {
  dibujar();
}
</script>
</body>
</html>
"""

# ===========================================================================
# UNA PAGINA, DOS DESTINOS
#
# La misma funcion construye el documento completo. Dentro del notebook se
# entrega con IPython.display.HTML; al disco se escribe tal cual. No hay dos
# rutas de renderizado que puedan divergir.
#
# En Colab cada salida de celda vive en un iframe aislado y el .ipynb guarda
# el HTML sin el estado del JavaScript: al reabrir el notebook guardado las
# figuras dejan de ser interactivas hasta reejecutar esta celda. El archivo
# descargable es el artefacto que sobrevive, y es el que va a la tesis.
# ===========================================================================


def construir_pagina(datos):
    return PLANTILLA_HTML.replace("__CDN__", CDN_ECHARTS).replace(
        "__DATOS__", json.dumps(datos, ensure_ascii=False))


PAGINA = construir_pagina(DATOS_FIGURAS)

with open(RUTA_HTML, "w", encoding="utf-8") as _fh:
    _fh.write(PAGINA)

import os as _os
print("=" * 74)
print("FIGURAS GENERADAS")
print("=" * 74)
print("  archivo: {}  ({:.0f} KB)".format(RUTA_HTML,
                                          _os.path.getsize(RUTA_HTML) / 1024))
print("  Este archivo es el artefacto que sobrevive al cierre del notebook.")
print("  Cada figura se exporta como SVG desde su barra de herramientas.")
if DATOS_FIGURAS["alertas"]:
    print()
    print("  ALERTAS")
    for _a in DATOS_FIGURAS["alertas"]:
        print("     ·", _a)
else:
    print()
    print("  Ninguna dimension supero los umbrales configurados.")
print("=" * 74)

try:
    from google.colab import files as _files
    _files.download(RUTA_HTML)
except Exception as _exc:
    print("(descarga automatica no disponible: {})".format(_exc))

display(HTML(PAGINA))

## 13. Descarga de resultados

Dos archivos, porque responden a dos preguntas distintas.

**`resultados_enriquecidos.csv` — el ancho.** Una fila por tuit: **todas las columnas
originales del archivo** más una columna por cada dimensión y método. Es el que abres en
Excel, el que le pasas a alguien, el que se mezcla con tu pipeline. Va en `utf-8-sig` para
que los acentos y el japonés se vean bien al abrirlo en Excel.

```
  id · created_at · lang · text · traducción · base · ... (las 52 originales)
  + atmosfera__nuevo          atmosfera__relevancia
    atmosfera__actual         atmosfera__traducido
    imagen_cultural__nuevo    imagen_cultural__relevancia
    perspectiva_politica__…   violence_salience__nuevo   safety_perceived__nuevo
  + p__atmosfera__pos / neg / na · p__violencia__violence / …   (si las pides)
```

**`resultados_largo.csv` — el largo.** Una fila por texto × método × dimensión. Es la forma
con la que se analiza y se grafica: filtras por `metodo`, agrupas por `lang_grupo` o por
`estrato`, y no tienes que adivinar qué significa cada nombre de columna.

### Dos cosas que el archivo ancho **no** trae, a propósito

**No hay ninguna columna que multiplique valencia por relevancia.** Es tentador —reduce todo
a un número— y reconstruye exactamente el defecto que el rediseño elimina: «habló y juzgó
neutro» (0.02 × 0.91) y «no habló del tema» (0.60 × 0.03) dan el mismo 0.018. Si necesitas
filtrar, filtra por `relevancia` y **después** lee la valencia; no las colapses.

**No hay una columna «resultado».** Cada dimensión trae su método explícito en el nombre
porque los tres miden cosas distintas: `__actual` es el léxico sobre el texto original,
`__traducido` el mismo léxico sobre la traducción, y `__nuevo` el zero-shot. Colapsarlos
escondería justo lo que este notebook existe para mostrar.

### Ojo con qué filas salen

Salen **las evaluadas**, no las del archivo original. Si dejaste `N_MUESTRA = 200` salen 200,
y la deduplicación por `id` ya descartó repetidas. La celda te imprime cuántas de cuántas, y
el pie del archivo queda anotado en la salida para que no haya que reconstruirlo de memoria.

In [ ]:
import pandas as pd

# ===========================================================================
# EXPORTACION DE RESULTADOS
#
# Sale de LARGO y de PROBS, no de `resultados`: el punto de extraccion sigue
# siendo uno solo. Dos formas, porque responden a dos preguntas distintas.
#
#   ancho  una fila por TEXTO, con las columnas originales del archivo mas
#          una columna por dimension y metodo. Para Excel y para compartir.
#   largo  una fila por texto x metodo x dimension. Para analizar.
#
# NINGUNA de las dos trae el producto valencia * relevancia. Ver la guardia al
# final: reintroduce el cero ambiguo que el rediseno elimina.
# ===========================================================================

SEP = "__"   # separador de nombres compuestos: dimension__metodo


def _tabla_ancha():
    # --- valencias: una columna por dimension y metodo ---------------------
    piv = (LARGO.pivot_table(index="texto_id", columns=["dimension", "metodo"],
                             values="valencia", aggfunc="first"))
    piv.columns = [SEP.join(c) for c in piv.columns]

    # --- relevancias: solo existen para el rediseno ------------------------
    rel = (LARGO[LARGO.metodo == "nuevo"]
           .pivot_table(index="texto_id", columns="dimension",
                        values="relevancia", aggfunc="first"))
    rel = rel.dropna(axis=1, how="all")
    rel.columns = [c + SEP + "relevancia" for c in rel.columns]

    ancha = piv.join(rel, how="outer")

    # --- probabilidades crudas, para que cada valor sea auditable ---------
    if EXPORTAR_PROBABILIDADES and len(PROBS):
        pr = PROBS.pivot_table(index="texto_id", columns=["grupo", "etiqueta"],
                               values="probabilidad", aggfunc="first")
        pr.columns = ["p" + SEP + SEP.join(c) for c in pr.columns]
        ancha = ancha.join(pr, how="outer")

    ancha = ancha.round(6).reset_index()

    # --- columnas de origen ------------------------------------------------
    if CORPUS_DF is not None and COLUMNAS.get("tweet_id"):
        # El archivo ENTERO, con sus 52 columnas, mas los resultados pegados.
        # Se une por identificador y no por posicion: el dedupe y el muestreo
        # ya movieron las filas de sitio.
        col_id = COLUMNAS["tweet_id"]
        origen = CORPUS_DF.copy()
        origen[col_id] = origen[col_id].astype(str)
        ancha = origen.merge(ancha, left_on=col_id, right_on="texto_id",
                             how="inner")
        ancha = ancha.drop(columns=["texto_id"])
        clave = col_id
    else:
        # Bateria o textos propios: no hay archivo de origen, asi que se
        # adjunta lo que el notebook sabe de cada texto.
        attrs = (LARGO[["texto_id", "texto", "frase", "lang", "objetivo"]]
                 .drop_duplicates("texto_id"))
        ancha = attrs.merge(ancha, on="texto_id", how="right")
        clave = "texto_id"
    return ancha, clave


TABLA_ANCHA, _CLAVE = _tabla_ancha()

# --- el largo, tal cual, sin el texto completo repetido en cada fila -------
_cols_largo = ["texto_id", "tweet_id", "created_at", "lang", "lang_grupo",
               "estrato", "modalidad", "frase", "objetivo",
               "metodo", "origen_texto", "dimension", "escala", "grupo",
               "valencia", "relevancia", "engagement_weight",
               "label_sentimiento"]
TABLA_LARGA = LARGO[[c for c in _cols_largo if c in LARGO.columns]].round(6)

# ---------------------------------------------------------------------------
# GUARDIA. La misma de los marcos, aplicada a lo que SALE del notebook, que es
# donde mas duele: un CSV se abre en Excel y ahi nadie recuerda por que no se
# podian multiplicar esas dos columnas.
#
#   hablo y juzgo neutro   0.02 * 0.91 = 0.018
#   no hablo del tema      0.60 * 0.03 = 0.018
# ---------------------------------------------------------------------------
_sospechosas = [c for c in TABLA_ANCHA.columns
                if any(k in str(c).lower() for k in
                       ("ponderad", "x_relev", "por_relev", "score_final",
                        "valencia_relev", "compuest"))]
assert not _sospechosas, (
    "La tabla ancha tiene columnas que parecen combinar valencia y relevancia: "
    "{}. Ese producto reintroduce el cero ambiguo; quitalas.".format(_sospechosas))

# --- escritura -------------------------------------------------------------
# utf-8-sig: sin el BOM, Excel abre los acentos y el japones como basura. El
# archivo de entrada venia asi por la misma razon.
TABLA_ANCHA.to_csv(RUTA_CSV_ANCHO, index=False, encoding="utf-8-sig")
_escritos = [RUTA_CSV_ANCHO]
if EXPORTAR_LARGO:
    TABLA_LARGA.to_csv(RUTA_CSV_LARGO, index=False, encoding="utf-8-sig")
    _escritos.append(RUTA_CSV_LARGO)

import os as _os

print("=" * 74)
print("RESULTADOS EXPORTADOS")
print("=" * 74)
for _f in _escritos:
    print("  {:<32} {:>8.1f} KB".format(_f, _os.path.getsize(_f) / 1024))
print()
print("  ancho:  {} filas x {} columnas   (clave: {})".format(
    len(TABLA_ANCHA), len(TABLA_ANCHA.columns), _CLAVE))
if CORPUS_DF is not None:
    print("          {} columnas originales del archivo + {} de resultados".format(
        len(CORPUS_DF.columns), len(TABLA_ANCHA.columns) - len(CORPUS_DF.columns)))
print("  largo:  {} filas x {} columnas".format(len(TABLA_LARGA), len(TABLA_LARGA.columns)))
print()
print("  columnas de resultados:")
for _c in [c for c in TABLA_ANCHA.columns if SEP in str(c)][:24]:
    print("     ", _c)
_resto = len([c for c in TABLA_ANCHA.columns if SEP in str(c)]) - 24
if _resto > 0:
    print("      ... y {} mas".format(_resto))
print()
print("  PROCEDENCIA: {}".format(PROCEDENCIA))
if PROCEDENCIA == "bateria":
    print("  Son los 30 textos de la BATERIA DE PRUEBA. No es una muestra de nada:")
    print("  ningun valor de este archivo es reportable como hallazgo sobre el corpus.")
elif PROCEDENCIA == "corpus":
    print("  Origen: {}".format(CORPUS_RUTA))
    print("  Salen las filas EVALUADAS ({}), no las del archivo original.".format(
        len(TABLA_ANCHA)))
    if N_MUESTRA:
        print("  N_MUESTRA = {} esta puesto: ponlo en None para el corpus completo.".format(
            N_MUESTRA))
    print("  Estos valores son DESCRIPTIVOS y NO estan validados contra")
    print("  codificacion humana. No los reportes como hallazgo sin eso.")
print()
print("  Recuerda: valencia y relevancia NO se multiplican. Filtra por")
print("  relevancia y DESPUES lee la valencia; colapsarlas devuelve el cero")
print("  ambiguo que este rediseno existe para separar.")
print("=" * 74)

try:
    from google.colab import files as _files
    for _f in _escritos:
        _files.download(_f)
except Exception as _e:
    print()
    print("(descarga automatica no disponible: {})".format(_e))
    print("Descargalos desde el panel de archivos de Colab, a la izquierda.")

## 14. Cómo leer esto

**En la sección 10** está el argumento. La misma frase, seis idiomas, y el método actual sale
**positivo en inglés** para un texto que dice que la policía era corrupta. La dispersión
—rango y desviación estándar— es el número que se reporta: mide cuánto varía el instrumento
cuando el contenido no varía. Idealmente sería cero. Y como esa dimensión no incorpora el
sentimiento, el número es exacto: no depende de la corrida.

**La columna `relevancia`** es información nueva que hoy no existe. Un `perspectiva_politica`
de 0.00 con relevancia 0.05 significa «el turista no habló de autoridades»; con relevancia
0.80 significa «habló de autoridades y su juicio fue neutro». Hoy ambos casos son el mismo
0.0000 y se promedian juntos.

**La vía traducida** responde la objeción obvia —«traduce todo y sigue con el léxico»— pero
no la cierra a favor de nadie: mide el léxico **junto con** la calidad de una traducción
automática. Si gana, parte del mérito es del traductor, y eso hay que decirlo.

**Sobre el corpus, cuidado con tres cosas:**

1. **No está validado.** Las columnas `tipo` y `justificacion` vienen vacías en las 2 631
   filas, así que no hay codificación humana contra la cual contrastar. Las figuras describen.
2. **Está dirigido por consulta.** Cuatro de cada diez filas vienen de búsquedas de
   victimización. Un `violence_salience` promedio sobre el corpus completo mide la consulta
   tanto como el fenómeno; por eso la figura 7 desglosa por estrato y las alertas lo gritan.
3. **Es 78% inglés y 16% japonés.** Los seis grupos de idioma del diseño original no existen
   aquí. Y el léxico no tiene una sola palabra en japonés: esos 432 tuits son invisibles para
   el método actual por construcción, no por falta de datos.

**Antes de adoptarlo hay que validar**, no basta con que se vea mejor:

1. Codifica a mano una muestra (≈200 tuits) en las tres dimensiones. **Este corpus no la
   trae**, así que es trabajo por hacer, no un paso que este notebook ya haya dado.
2. Calcula el alfa de Krippendorff de cada método contra tus códigos —
   la infraestructura ya está en `wc2026_analyzer/validation/agreement.py`, con umbral 0.67.
3. Adopta el rediseño solo si gana. Si empata, gana igual por el argumento del sesgo
   entre idiomas, pero eso hay que poder decirlo con el número en la mano.

**Y mide el tiempo.** Son 13 pasadas de un modelo *large* por tuit; sobre el corpus completo,
34 203. Corre primero con `N_MUESTRA = 200`, mira el tiempo por texto que imprime la celda de evaluación
y multiplica antes de comprometerte. En CPU, que es como está configurado `Dockerfile.nlp`,
puede ser inviable.